# TP Qualité des données IRVE

## Objectif
Auditer, nettoyer et contrôler la qualité du fichier consolidé des bornes de recharge pour véhicules électriques.

## Étapes
1. Chargement et exploration
2. Audit qualité initial
3. Étude du grain et des doublons
4. Contrôle des coordonnées GPS
5. Nettoyage et enrichissement
6. Validation automatisée
7. Monitoring

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import re
import numpy as np
import pandas as pd
import requests
import folium

from tqdm.auto import tqdm
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from pathlib import Path


tqdm.pandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Imports réussis")

Imports réussis


# Partie 1 — Profiling et audit initial

## Objectif

Cette première partie consiste à observer et mesurer la qualité du dataset IRVE avant toute transformation.

Les analyses réalisées portent sur :

* les dimensions du dataset ;
* les noms et les types des colonnes ;
* les valeurs manquantes ;
* les doublons stricts et les identifiants répétés ;
* les coordonnées géographiques douteuses ;
* les puissances aberrantes ;
* les dates absentes, invalides ou futures ;
* les six dimensions de la qualité des données ;
* le calcul d’un score qualité initial.

À ce stade, aucune donnée n’est supprimée ou modifiée. Le score obtenu servira de référence pour comparer la qualité avant et après nettoyag


In [2]:
FILE_PATH = "data/consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv"


OUTPUT_DIR = Path("outputs_final")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Dossier de sortie :", OUTPUT_DIR)
df = pd.read_csv(
    FILE_PATH,
    low_memory=False
)

print("Dimensions :", df.shape)

Dossier de sortie : outputs_final
Dimensions : (227232, 52)


## Description des colonnes du dataset IRVE

Le dataset contient **52 colonnes** organisées autour de plusieurs niveaux d’information : les acteurs, les stations, les points de charge, les caractéristiques techniques, les modalités d’accès, la traçabilité et les données consolidées.

### 1. Informations sur les acteurs

* `nom_amenageur` : nom du responsable ou propriétaire de l’installation.
* `siren_amenageur` : numéro SIREN de l’aménageur.
* `contact_amenageur` : contact de l’aménageur.
* `nom_operateur` : nom de l’opérateur qui exploite la borne.
* `contact_operateur` : contact de l’opérateur.
* `telephone_operateur` : numéro de téléphone de l’opérateur.
* `nom_enseigne` : nom commercial visible par l’utilisateur.

### 2. Informations sur la station

Une station peut contenir plusieurs points de charge.

* `id_station_itinerance` : identifiant interopérable de la station.
* `id_station_local` : identifiant interne de la station.
* `nom_station` : nom de la station.
* `implantation_station` : type d’implantation de la station.
* `adresse_station` : adresse postale.
* `code_insee_commune` : code INSEE de la commune.
* `coordonneesXY` : coordonnées géographiques brutes.
* `nbre_pdc` : nombre de points de charge déclarés dans la station.

### 3. Informations sur le point de charge

* `id_pdc_itinerance` : identifiant interopérable du point de charge.
* `id_pdc_local` : identifiant local du point de charge.
* `puissance_nominale` : puissance maximale du point de charge, normalement exprimée en kW.

### 4. Types de prises

Ces colonnes indiquent si le point de charge propose un type de connecteur donné.

* `prise_type_ef` : prise domestique de type E/F.
* `prise_type_2` : connecteur de type 2.
* `prise_type_combo_ccs` : connecteur Combo CCS.
* `prise_type_chademo` : connecteur CHAdeMO.
* `prise_type_autre` : autre type de prise.
* `cable_t2_attache` : présence d’un câble de type 2 attaché.

### 5. Accès, paiement et utilisation

* `gratuit` : recharge gratuite ou payante.
* `paiement_acte` : paiement ponctuel possible.
* `paiement_cb` : paiement par carte bancaire possible.
* `paiement_autre` : autre moyen de paiement.
* `tarification` : description de la tarification.
* `condition_acces` : conditions d’accès à la station.
* `reservation` : possibilité de réserver.
* `horaires` : horaires d’ouverture.
* `accessibilite_pmr` : accessibilité aux personnes à mobilité réduite.
* `restriction_gabarit` : restrictions de hauteur, longueur ou poids.
* `station_deux_roues` : station adaptée aux deux-roues.

### 6. Raccordement électrique

* `raccordement` : type de raccordement au réseau électrique.
* `num_pdl` : numéro du point de livraison électrique.

### 7. Dates et observations

* `date_mise_en_service` : date de mise en service du point de charge.
* `observations` : remarques complémentaires.
* `date_maj` : date de mise à jour déclarée.
* `last_modified` : date de dernière modification de la ressource.
* `created_at` : date de création de la ligne dans le système de consolidation.

### 8. Traçabilité Data.gouv

* `datagouv_dataset_id` : identifiant du jeu de données source.
* `datagouv_resource_id` : identifiant de la ressource source.
* `datagouv_organization_or_owner` : organisation ayant publié les données.

### 9. Colonnes consolidées

* `consolidated_longitude` : longitude consolidée.
* `consolidated_latitude` : latitude consolidée.
* `consolidated_code_postal` : code postal consolidé.
* `consolidated_commune` : commune consolidée.
* `consolidated_is_lon_lat_correct` : indicateur de validité des coordonnées.
* `consolidated_is_code_insee_verified` : indique si le code INSEE a été vérifié.
* `consolidated_is_code_insee_modified` : indique si le code INSEE a été corrigé.




## 2. Qualité des données

Cette section permet d'identifier les principaux problèmes de qualité avant tout nettoyage :

- les valeurs manquantes ;
- les types de données ;
- les coordonnées géographiques douteuses ;
- les puissances aberrantes ;
- les dates absentes, invalides ou futures.


In [3]:
# Calcul du nombre et du pourcentage de valeurs manquantes par colonne

audit_manquants = pd.DataFrame({
    "nombre_manquants": df.isna().sum(),
    "pourcentage_manquants": (df.isna().mean() * 100).round(2)
})

audit_manquants = audit_manquants.sort_values(
    by="pourcentage_manquants",
    ascending=False
)

audit_manquants.head(52)

,nombre_manquants,pourcentage_manquants
observations,180712,79.53
tarification,170887,75.20
num_pdl,111260,48.96
cable_t2_attache,106741,46.97
consolidated_code_postal,92870,40.87
date_mise_en_service,92271,40.61
siren_amenageur,90985,40.04
raccordement,79307,34.90
id_pdc_local,78678,34.62
consolidated_commune,76393,33.62


Comme le tableau est trié du plus grand au plus petit pourcentage de valeurs manquantes, les 20 premières lignes montrent directement les colonnes les plus problématiques.

In [4]:
audit_manquants.head(20)

,nombre_manquants,pourcentage_manquants
observations,180712,79.53
tarification,170887,75.20
num_pdl,111260,48.96
cable_t2_attache,106741,46.97
consolidated_code_postal,92870,40.87
date_mise_en_service,92271,40.61
siren_amenageur,90985,40.04
raccordement,79307,34.90
id_pdc_local,78678,34.62
consolidated_commune,76393,33.62


### 2.2 Analys des types de données

Cette étape permet de vérifier le type de chaque colonne et le nombre de valeurs distinctes.

L'objectif est d'identifier notamment :

- les dates stockées comme texte ;
- les codes postaux et les identifiants stockés comme nombres ;
- les colonnes booléennes contenant plusieurs formats ;
- les variables catégorielles présentant un nombre limité de modalités.


In [5]:
# Types des colonnes
types_colonnes = pd.DataFrame({
    "type": df.dtypes,
    "nombre_valeurs_uniques": df.nunique(dropna=False)
})

display(types_colonnes)

,type,nombre_valeurs_uniques
nom_amenageur,object,4409
siren_amenageur,float64,2941
contact_amenageur,object,909
nom_operateur,object,359
contact_operateur,object,356
telephone_operateur,object,672
nom_enseigne,object,6985
id_station_itinerance,object,63946
id_station_local,object,55464
nom_station,object,45977


### Interprétation des types

L'analyse met en évidence plusieurs points de vigilance :

- les colonnes `date_mise_en_service`, `date_maj`, `last_modified` et `created_at` sont stockées comme texte ;
- `consolidated_code_postal` est stocké comme nombre décimal alors qu'un code postal doit être traité comme du texte ;
- `siren_amenageur` est également stocké comme nombre alors qu'il s'agit d'un identifiant ;
- plusieurs colonnes attendues comme booléennes présentent plusieurs formes de valeurs ;
- les indicateurs `consolidated_is_*` sont correctement reconnus comme booléens.

Ces constats concernent principalement les dimensions de validité et de cohérence.


In [6]:
# Contrôle des coordonnées signalées comme correctes ou incorrectes

controle_coordonnees = (
    df["consolidated_is_lon_lat_correct"]
    .value_counts(dropna=False)
)

display(controle_coordonnees)

consolidated_is_lon_lat_correct
True     145508
False     81724
Name: count, dtype: int64

### Résultat du contrôle des coordonnées

Sur les 227 232 lignes du dataset :

- 145 508 lignes ont des coordonnées considérées comme correctes ;
- 81 724 lignes ont des coordonnées signalées comme incorrectes.

Cela représente environ 35,96 % du dataset.

Ce résultat concerne principalement la dimension d'exactitude.

In [7]:
puissance = pd.to_numeric(
    df["puissance_nominale"],
    errors="coerce"
)

mask_puissance_aberrante = (
    (puissance <= 0)
    | (puissance > 1000)
)

print("Puissances manquantes :", puissance.isna().sum())
print("Puissances inférieures ou égales à 0 :", (puissance <= 0).sum())
print("Puissances supérieures à 1000 kW :", (puissance > 1000).sum())
print("Total des puissances aberrantes :", mask_puissance_aberrante.sum())

Puissances manquantes : 0
Puissances inférieures ou égales à 0 : 5282
Puissances supérieures à 1000 kW : 975
Total des puissances aberrantes : 6257


### Résultat du contrôle des puissances

La colonne `puissance_nominale` ne contient aucune valeur manquante.

6 257 valeurs sont considérées comme aberrantes selon la règle utilisée :

- 5 282 valeurs sont inférieures ou égales à 0 kW ;
- 975 valeurs sont supérieures à 1 000 kW.

Ces anomalies concernent principalement la dimension de validité.

Piste probable : puissances très élevées peuvent correspondre à une erreur d'unité (W vs kW)

In [8]:
# Contrôle des colonnes de dates

colonnes_dates = [
    "date_mise_en_service",
    "date_maj",
    "last_modified",
    "created_at"
]

for colonne in colonnes_dates:
    dates = pd.to_datetime(
        df[colonne],
        errors="coerce",
        utc=True
    )

    print(f"\n--- {colonne} ---")
    print("Valeurs absentes ou invalides :", dates.isna().sum())
    print("Date minimale :", dates.min())
    print("Date maximale :", dates.max())


--- date_mise_en_service ---
Valeurs absentes ou invalides : 92271
Date minimale : 1900-01-01 00:00:00+00:00
Date maximale : 2026-06-24 00:00:00+00:00

--- date_maj ---
Valeurs absentes ou invalides : 292
Date minimale : 2012-09-10 00:00:00+00:00
Date maximale : 2026-12-30 00:00:00+00:00

--- last_modified ---
Valeurs absentes ou invalides : 33877
Date minimale : 2024-02-09 06:22:40.878000+00:00
Date maximale : 2026-06-24 21:00:21.100000+00:00

--- created_at ---
Valeurs absentes ou invalides : 0
Date minimale : 2021-05-10 21:01:29.234000+00:00
Date maximale : 2026-06-24 15:14:32.641000+00:00


### Résultat du contrôle des dates

L'analyse des colonnes de dates met en évidence plusieurs problèmes de qualité.

#### `date_mise_en_service`

- 92 271 valeurs sont absentes ou non convertibles ;
- la date minimale est le 1er janvier 1900 ;
- la date maximale est le 24 juin 2026.

La date de 1900 semble peu crédible pour une borne de recharge électrique. Elle peut correspondre à une valeur par défaut ou à une erreur de saisie.

#### `date_maj`

- 292 valeurs sont absentes ou invalides ;
- la date minimale est le 10 septembre 2012 ;
- la date maximale est le 30 décembre 2026.

La date maximale est future par rapport à la date du fichier. Elle doit donc être signalée comme anomalie de cohérence ou de fraîcheur.

#### `last_modified`

- 33 877 valeurs sont absentes ou invalides ;
- les dates observées vont du 9 février 2024 au 24 juin 2026.

Cette colonne décrit la dernière modification de la ressource source sur Data.gouv.

#### `created_at`

- aucune valeur n'est absente ;
- les dates vont du 10 mai 2021 au 24 juin 2026.

Cette colonne est la plus complète parmi les colonnes de dates analysées.

### Dimensions concernées

- Complétude : dates absentes ;
- Validité : dates non convertibles ou peu plausibles ;
- Cohérence : dates futures ou dates par défaut ;
- Fraîcheur : dates de mise à jour anciennes.

In [9]:
aujourdhui = pd.Timestamp.now(tz="UTC")

for colonne in colonnes_dates:
    dates = pd.to_datetime(
        df[colonne],
        errors="coerce",
        utc=True
    )

    print(
        colonne,
        "- dates futures :",
        (dates > aujourdhui).sum()
    )

date_mise_en_service - dates futures : 0
date_maj - dates futures : 56
last_modified - dates futures : 0
created_at - dates futures : 0


### Dates futures

La colonne `date_maj` contient 56 dates postérieures à la date actuelle.

Ces valeurs sont considérées comme des anomalies de cohérence temporelle et seront vérifiées lors du nettoyage.

In [10]:
#1 grille_audit
grille_audit = pd.DataFrame([
    {
        "Dimension": "Complétude",
        "Problème": "Valeurs manquantes",
        "Colonne": "Plusieurs colonnes",
        "Résultat": "Présence importante de valeurs manquantes",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Unicité",
        "Problème": "Identifiants répétés",
        "Colonne": "id_pdc_itinerance",
        "Résultat": "67 094 occurrences supplémentaires",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Exactitude",
        "Problème": "Coordonnées douteuses",
        "Colonne": "consolidated_is_lon_lat_correct",
        "Résultat": "81 724 lignes, soit 35,96 %",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Validité",
        "Problème": "Puissances aberrantes",
        "Colonne": "puissance_nominale",
        "Résultat": "6 257 valeurs",
        "Gravité": "Élevée"
    },
    {
        "Dimension": "Cohérence",
        "Problème": "Types et formats non homogènes",
        "Colonne": "Dates, booléens, adresses",
        "Résultat": "Dates en texte et problèmes d'encodage",
        "Gravité": "Moyenne"
    },
    {
        "Dimension": "Fraîcheur",
        "Problème": "Dates futures",
        "Colonne": "date_maj",
        "Résultat": "56 dates futures",
        "Gravité": "Moyenne"
    }
])

display(grille_audit)

,Dimension,Problème,Colonne,Résultat,Gravité
0,Complétude,Valeurs manquantes,Plusieurs colonnes,Présence importante de valeurs manquantes,Élevée
1,Unicité,Identifiants répétés,id_pdc_itinerance,67 094 occurrences supplémentaires,Élevée
2,Exactitude,Coordonnées douteuses,consolidated_is_lon_lat_correct,"81 724 lignes, soit 35,96 %",Élevée
3,Validité,Puissances aberrantes,puissance_nominale,6 257 valeurs,Élevée
4,Cohérence,Types et formats non homogènes,"Dates, booléens, adresses",Dates en texte et problèmes d'encodage,Moyenne
5,Fraîcheur,Dates futures,date_maj,56 dates futures,Moyenne


### Interprétation de la grille d'audit

L'audit met en évidence des problèmes sur les six dimensions de la qualité des données.

Les anomalies les plus importantes concernent :

- l'unicité, avec 67 094 occurrences supplémentaires de `id_pdc_itinerance` ;
- l'exactitude, avec 81 724 coordonnées signalées comme douteuses ;
- la validité, avec 6 257 puissances aberrantes ;
- la complétude, avec plusieurs colonnes fortement incomplètes.

Les problèmes de cohérence et de fraîcheur sont également présents, notamment à cause des formats non homogènes, des problèmes d'encodage et des dates futures.

## Score qualité initial

Le score initial est calculé sur les six dimensions de la qualité des données.

Chaque dimension est notée sur 100, puis une moyenne simple est calculée. Ce score sert de point de comparaison avec le score obtenu après nettoyage.

In [11]:
# 1. Complétude : pourcentage de cellules non manquantes
score_completude = df.notna().mean().mean() * 100


# 2. Unicité : proportion d'identifiants de points de charge uniques
score_unicite = (
    df["id_pdc_itinerance"].nunique(dropna=True)
    / len(df)
) * 100


# 3. Exactitude : proportion de coordonnées signalées comme correctes
score_exactitude = (
    df["consolidated_is_lon_lat_correct"].mean()
) * 100


# 4. Validité : proportion de puissances comprises entre 0 et 1000 kW
score_validite = (
    ~mask_puissance_aberrante
).mean() * 100


# 5. Fraîcheur : date_maj présente et non future
dates_maj = pd.to_datetime(
    df["date_maj"],
    errors="coerce",
    utc=True
)

aujourdhui = pd.Timestamp.now(tz="UTC")

score_fraicheur = (
    dates_maj.notna()
    & (dates_maj <= aujourdhui)
).mean() * 100

In [12]:
colonnes_booleennes = [
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "gratuit",
    "paiement_acte",
    "paiement_cb",
    "reservation",
    "cable_t2_attache"
]

valeurs_booleennes_valides = {
    "true", "false",
    "vrai", "faux",
    "1", "0"
}

scores_coherence = []

for colonne in colonnes_booleennes:
    valeurs_normalisees = (
        df[colonne]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    valeurs_coherentes = (
        valeurs_normalisees.isin(valeurs_booleennes_valides)
        | valeurs_normalisees.isna()
    )

    scores_coherence.append(valeurs_coherentes.mean() * 100)

score_coherence = sum(scores_coherence) / len(scores_coherence)

In [13]:
scores_initiaux = pd.Series({
    "Complétude": score_completude,
    "Validité": score_validite,
    "Cohérence (booléens)": score_coherence,
    "Unicité": score_unicite,
    "Exactitude": score_exactitude,
    "Fraîcheur": score_fraicheur
}).round(2)

score_qualite_initial = scores_initiaux.mean().round(2)

display(scores_initiaux.to_frame("Score sur 100"))

print(
    "Score qualité global initial :",
    score_qualite_initial,
    "/ 100"
)

,Score sur 100
Complétude,87.73
Validité,97.25
Cohérence (booléens),100.00
Unicité,70.47
Exactitude,64.03
Fraîcheur,99.85


Score qualité global initial : 86.56 / 100


## Justification des scores de qualité

### Complétude — 87,73/100

Ce score correspond au pourcentage moyen de cellules renseignées dans l’ensemble du dataset.

Il est relativement élevé, mais plusieurs colonnes restent fortement incomplètes, notamment :

* `observations` ;
* `tarification` ;
* `num_pdl` ;
* `consolidated_code_postal` ;
* `date_mise_en_service`.

Le score montre donc que la majorité des données sont présentes, mais que certaines informations importantes manquent encore.

### Validité — 97,25/100

Ce score mesure la proportion de puissances considérées comme valides selon la règle suivante :

* puissance strictement supérieure à 0 kW ;
* puissance inférieure ou égale à 1 000 kW.

Sur 227 232 lignes, 6 257 puissances ne respectent pas cette règle :

* 5 282 sont inférieures ou égales à 0 ;
* 975 sont supérieures à 1 000 kW.

Le score reste élevé, car la grande majorité des puissances se situe dans l’intervalle retenu.

Cette mesure doit toutefois être interprétée avec prudence, car certaines valeurs supérieures à 1 000 peuvent être exprimées en watts au lieu de kilowatts.

### Cohérence — 100/100

Ce score a été calculé uniquement à partir de la normalisation des colonnes supposées booléennes.

Après conversion en minuscules et suppression des espaces, les valeurs observées ont été considérées comme compatibles avec les formes autorisées :

* `true` ;
* `false` ;
* `vrai` ;
* `faux` ;
* `1` ;
* `0` ;
* valeurs manquantes.

Selon cette règle technique, toutes les valeurs contrôlées sont cohérentes, ce qui explique le score de 100/100.

Cependant, ce score est trop optimiste pour représenter toute la cohérence du dataset. D’autres anomalies ont été observées :

* caractères mal encodés dans les adresses ;
* différences de casse ;
* variantes d’écriture des communes et des stations ;
* informations contradictoires entre plusieurs sources ;
* puissances différentes pour un même identifiant.

Le score de cohérence mesure donc seulement un aspect limité de cette dimension. Il ne signifie pas que l’ensemble du dataset est parfaitement cohérent.

### Unicité — 70,47/100

Ce score correspond au rapport entre le nombre d’identifiants `id_pdc_itinerance` uniques et le nombre total de lignes.

Le dataset contient :

* 227 232 lignes ;
* 160 138 identifiants uniques ;
* 67 094 occurrences supplémentaires.

Le score est donc dégradé par le nombre important d’identifiants répétés.

Cependant, ces répétitions ne sont pas nécessairement des erreurs. Elles peuvent représenter plusieurs versions ou publications d’un même point de charge. Le score mesure ici la répétition des identifiants, mais pas encore la légitimité de chaque répétition.

### Exactitude — 64,03/100

Ce score correspond à la proportion de lignes pour lesquelles la colonne `consolidated_is_lon_lat_correct` vaut `True`.

Le dataset contient :

* 145 508 coordonnées considérées comme correctes ;
* 81 724 coordonnées signalées comme douteuses.

L’exactitude est donc la dimension la plus faible.

Ce score repose sur un indicateur déjà présent dans le fichier consolidé. Il signale un risque, mais ne permet pas à lui seul de savoir si chaque coordonnée est réellement fausse.

### Fraîcheur — 99,85/100

Ce score repose sur la colonne `date_maj`.

Une ligne est considérée comme fraîche lorsque :

* la date est présente ;
* elle peut être convertie en date ;
* elle n’est pas située dans le futur.

Le score est très élevé, car seules quelques lignes présentent une date manquante, invalide ou future.

Le dataset contient notamment 56 dates futures dans `date_maj`.

Cette mesure ne garantit toutefois pas que toutes les données sont réellement récentes. Une date valide mais ancienne peut toujours indiquer une information peu fraîche.

## Conclusion

Les scores permettent de résumer rapidement la qualité initiale du dataset, mais chacun dépend d’une règle de calcul précise.

Ils doivent donc être accompagnés d’une analyse détaillée, car un score élevé peut masquer certains problèmes non pris en compte par la règle utilisée.


## Conclusion du profiling et de l’audit initial

À ce stade, aucune transformation ni aucun nettoyage n’a été appliqué au dataset.

Le travail réalisé correspond uniquement à une phase d’audit, d’observation et de mesure. Nous avons :

* chargé le fichier CSV ;
* analysé les 52 colonnes ;
* compté les valeurs manquantes ;
* repéré les identifiants répétés ;
* contrôlé la qualité des coordonnées géographiques ;
* vérifié les puissances nominales ;
* examiné les colonnes de dates ;
* calculé plusieurs indicateurs de qualité.

Le score qualité global initial obtenu est de **86,56/100**.

Ce score représente une estimation de la qualité actuelle de la base avant nettoyage.
Il ne constitue pas une vérité absolue, car il dépend directement des règles et des seuils choisis pour chaque dimension :

* la complétude dépend du taux de cellules renseignées ;
* l’unicité dépend du nombre d’identifiants uniques ;
* l’exactitude dépend du contrôle des coordonnées ;
* la validité dépend des seuils retenus pour les puissances ;
* la fraîcheur dépend de la qualité de la colonne `date_maj` ;
* la cohérence dépend des règles appliquées aux formats et aux valeurs booléennes.

Le score de cohérence de 100/100 doit être interprété avec prudence. 
Des problèmes d’encodage, de casse, de formats et des différences entre sources ont été observés.
Cela montre qu’un score synthétique peut masquer certaines anomalies si les règles de calcul sont trop simples.

Ce score initial servira donc de référence pour mesurer l’amélioration de la qualité après les étapes de nettoyage et de transformation.


## Rapport automatique avec ydata-profiling

Un rapport automatique est généré afin d’obtenir une vue globale des types de données, des valeurs manquantes, des distributions, des doublons et des principales alertes qualité.

Le mode minimal est utilisé afin de limiter le temps de calcul sur ce dataset volumineux.

In [14]:
from ydata_profiling import ProfileReport

print("ydata-profiling importé avec succès")

ydata-profiling importé avec succès


C:\Users\Sara\AppData\Local\Temp\ipykernel_6404\2161875287.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [15]:
echantillon_profiling = df.sample(
    n=30000,
    random_state=42
)

profile = ProfileReport(
    echantillon_profiling,
    title="Profiling initial IRVE - échantillon de 30 000 lignes",
    minimal=True
)

chemin_rapport_profiling = OUTPUT_DIR / "rapport_profiling_irve_initial.html"

profile.to_file(chemin_rapport_profiling)

print("Rapport généré :", chemin_rapport_profiling)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 52/52 [00:03<00:00, 13.28it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Rapport généré : outputs_final\rapport_profiling_irve_initial.html


# Partie 2 — Étude du grain et des doublons

L'objectif est de comprendre pourquoi certains `id_pdc_itinerance` apparaissent plusieurs fois avant de définir une règle de dédoublonnage.

## 1. Compréhension du grain

L'objectif est de comprendre ce que représente une ligne avant de supprimer des données.

Une ligne semble représenter un point de charge provenant d'une ressource ou d'une livraison donnée.

Un même `id_pdc_itinerance` peut donc apparaître plusieurs fois sans que les lignes soient strictement identiques.


In [16]:
# Nombre de doublons stricts
doublons_stricts = df.duplicated().sum()

# Nombre d'identifiants uniques
nombre_ids_uniques = df["id_pdc_itinerance"].nunique()

# Nombre d'identifiants apparaissant plusieurs fois
comptage_ids = df["id_pdc_itinerance"].value_counts()

nombre_ids_repetes = (comptage_ids > 1).sum()

# Nombre de lignes appartenant à des groupes répétés
lignes_ids_repetes = comptage_ids[comptage_ids > 1].sum()

# Nombre de lignes en trop si on gardait une ligne par identifiant
lignes_potentiellement_en_trop = (
    comptage_ids[comptage_ids > 1] - 1
).sum()

print("Doublons stricts :", doublons_stricts)
print("Identifiants uniques :", nombre_ids_uniques)
print("Identifiants répétés :", nombre_ids_repetes)
print("Lignes appartenant à des groupes répétés :", lignes_ids_repetes)
print("Lignes potentiellement en trop :", lignes_potentiellement_en_trop)

Doublons stricts : 0
Identifiants uniques : 160138
Identifiants répétés : 63870
Lignes appartenant à des groupes répétés : 130964
Lignes potentiellement en trop : 67094


Un identifiant répété est une anomalie à analyser, mais ce n’est pas automatiquement une erreur. Il faut comparer les dates, les sources et les caractéristiques des lignes avant de décider.

In [17]:
# Afficher les 10 identifiants les plus répétés
comptage_ids_repetes = comptage_ids[comptage_ids > 1]

display(comptage_ids_repetes.head(10))

id_pdc_itinerance
Non concerné       118
FRLIBE3126085        4
FRCG0E002113         3
FRCG0E002114         3
FRCG0E002115         3
FRCG0E002116         3
FRCG0E001353         3
FRALLEGO9004771      3
FRALLEGO9004772      3
FRALLEGO9004773      3
Name: count, dtype: int64

In [18]:
# Garder uniquement les vrais identifiants répétés
comptage_ids_repetes_valides = comptage_ids[
    (comptage_ids > 1)
    & (comptage_ids.index != "Non concerné")
]

display(comptage_ids_repetes_valides.head(10))

id_pdc_itinerance
FRLIBE3126085      4
FRCG0E002113       3
FRCG0E002114       3
FRCG0E002115       3
FRCG0E002116       3
FRCG0E001353       3
FRALLEGO9004771    3
FRALLEGO9004772    3
FRALLEGO9004773    3
FRCG0E001355       3
Name: count, dtype: int64

In [19]:
id_exemple = comptage_ids_repetes_valides.index[0]

colonnes_analyse = [
    "id_pdc_itinerance",
    "id_station_itinerance",
    "nom_station",
    "adresse_station",
    "puissance_nominale",
    "date_maj",
    "last_modified",
    "datagouv_resource_id",
    "datagouv_organization_or_owner",
    "consolidated_longitude",
    "consolidated_latitude"
]

display(
    df.loc[
        df["id_pdc_itinerance"] == id_exemple,
        colonnes_analyse
    ].sort_values("date_maj")
)

,id_pdc_itinerance,id_station_itinerance,nom_station,adresse_station,puissance_nominale,date_maj,last_modified,datagouv_resource_id,datagouv_organization_or_owner,consolidated_longitude,consolidated_latitude
105427,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de lâÃÃ´Orge 91220 Brtigny-sur-Orge,22.0,2023-10-11,2026-01-09T13:01:51.509000+00:00,7514352c-c1ad-4530-8dc5-bf05bac9a786,eoliberty,2.297193,48.618492
105428,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de l‚Äö√Ñ√¥Orge 91220 Brétigny-sur-Orge,22.0,2023-10-11,2025-11-24T21:47:43.507000+00:00,04b89d29-2904-42c9-9e6b-f3c7dcd988ec,greenea,2.297193,48.618492
105425,FRLIBE3126085,FRLIBE3126085,Résidence Carouge,34 rue de l'Orge,7.4,2023-10-21,2024-12-06T06:47:29.008000+00:00,2f2c05b7-3ea6-461b-a6ab-7d5a3fed0718,bornevo,2.300000,48.620000
105426,FRLIBE3126085,FRLIBE3126085,Residence Carouge,34 Rue de l’Orge 91220 Brétigny-sur-Orge,22.0,2023-11-10,2024-12-06T06:47:51.017000+00:00,26160785-4f6d-428e-bcbb-9172278919ed,charge-in,2.297193,48.618492


In [20]:
# Sous-ensemble des lignes avec un véritable identifiant répété
ids_repetes_valides = comptage_ids_repetes_valides.index

df_repetes = df[
    df["id_pdc_itinerance"].isin(ids_repetes_valides)
].copy()

# Analyse globale par identifiant
analyse_groupes = df_repetes.groupby("id_pdc_itinerance").agg(
    nombre_lignes=("id_pdc_itinerance", "size"),
    nombre_sources=("datagouv_resource_id", "nunique"),
    nombre_organisations=("datagouv_organization_or_owner", "nunique"),
    nombre_puissances=("puissance_nominale", "nunique"),
    nombre_longitudes=("consolidated_longitude", "nunique"),
    nombre_latitudes=("consolidated_latitude", "nunique"),
    nombre_dates_maj=("date_maj", "nunique")
)

display(analyse_groupes.head())

,nombre_lignes,nombre_sources,nombre_organisations,nombre_puissances,nombre_longitudes,nombre_latitudes,nombre_dates_maj
id_pdc_itinerance,,,,,,,
ATHTBE1012062,2,2,2,2,2,2,2
ATHTBE1012063,2,2,2,2,2,2,2
FR190E190001SPE24ABB360121,2,2,2,1,2,2,2
FR190E190001SPE24ABB360122,2,2,2,1,2,2,2
FR190E190001SPE24ABB360341,2,2,2,1,2,2,2


In [21]:
resume_doublons = {
    "Groupes répétés analysés": len(analyse_groupes),
    "Groupes avec plusieurs sources": (analyse_groupes["nombre_sources"] > 1).sum(),
    "Groupes avec plusieurs organisations": (analyse_groupes["nombre_organisations"] > 1).sum(),
    "Groupes avec plusieurs puissances": (analyse_groupes["nombre_puissances"] > 1).sum(),
    "Groupes avec plusieurs longitudes": (analyse_groupes["nombre_longitudes"] > 1).sum(),
    "Groupes avec plusieurs latitudes": (analyse_groupes["nombre_latitudes"] > 1).sum(),
    "Groupes avec plusieurs dates de mise à jour": (analyse_groupes["nombre_dates_maj"] > 1).sum()
}

for cle, valeur in resume_doublons.items():
    print(f"{cle} : {valeur}")

Groupes répétés analysés : 63869
Groupes avec plusieurs sources : 63869
Groupes avec plusieurs organisations : 63869
Groupes avec plusieurs puissances : 9719
Groupes avec plusieurs longitudes : 32861
Groupes avec plusieurs latitudes : 33643
Groupes avec plusieurs dates de mise à jour : 59716


### Conclusion provisoire sur les identifiants répétés

L'analyse montre que les répétitions peuvent correspondre à :

- plusieurs publications provenant de sources différentes ;
- plusieurs versions avec des dates de mise à jour différentes ;
- des conflits sur la puissance ;
- des écarts de coordonnées ;
- des organisations productrices différentes.

Ces répétitions ne doivent donc pas être supprimées automatiquement. La règle de dédoublonnage sera définie à partir de ces résultats globaux.


## 2. Définition de la règle de dédoublonnage

L'analyse montre que les identifiants répétés proviennent systématiquement de plusieurs ressources et organisations.

La règle retenue est la suivante :

- conserver les lignes dont l'identifiant n'est pas répété ;
- pour les identifiants répétés sans conflit majeur, conserver la version ayant la `date_maj` la plus récente ;
- en cas d'égalité sur `date_maj`, utiliser `last_modified` comme second critère ;
- isoler les groupes présentant plusieurs puissances ou des coordonnées différentes afin de conserver une trace des conflits.

Cette règle évite un `drop_duplicates()` aveugle.

Quel est le problème ?

Le fichier contient plusieurs lignes avec le même id_pdc_itinerance
Mais ces lignes ne sont pas identiques.

=>plusieurs sources publient des informations sur leMEME point de charge.

pas des doublons stricts. mais plusieurs versions ou publications duMEME objet

Pourquoi faut-il dédoublonner ?
1 point de charge = 1 ligne
choisir quelle version conserver

La règle la plus simple

La règle la plus facile à comprendre et à défendre est :

Pour chaque id_pdc_itinerance, conserver la ligne ayant la date_maj la plus récente

Pourquoi utiliser last_modified ?

Parfois, deux lignes du même identifiant peuvent avoir exactement la même date_maj

Pourquoi conserver une table de conflits ?

Certains identifiants répétés contiennent des informations contradictoires : plusieurs puissances oordonnées et adresses

garde les lignes contradictoires dans une table séparée pour la tracabilité 
ATTENTION LA LIGNE RETENU PEUT AVOIR DES ERREURS !!!!


Resumer
Même identifiant répété =>Comparer les dates=> garder la date_maj la plus récente
      
Si égalité : garder le last_modified le plus récent
        
PS : Conserver les cas contradictoires dans une table de conflits

La décision métier centrale est donc :

Une ligne finale représente la version la plus récente connue d’un point de charge.

# Sauvegarder la version brute

In [22]:
# Sauvegarde du dataset avant dédoublonnage
df_brut = df.copy()

print("Nombre de lignes avant traitement :", len(df_brut))

Nombre de lignes avant traitement : 227232


# Convertir les dates

In [23]:
# Conversion des dates utilisées pour choisir la version la plus récente

df["date_maj"] = pd.to_datetime(
    df["date_maj"],
    errors="coerce",
    utc=True
)

df["last_modified"] = pd.to_datetime(
    df["last_modified"],
    errors="coerce",
    utc=True
)

pour pouvoir les trier correctementmmmmmmmm

# Mettre de côté les identifiants conflictuels

In [24]:
# Identifiants présentant des informations contradictoires

ids_conflit = analyse_groupes.loc[
    (analyse_groupes["nombre_puissances"] > 1)
    | (analyse_groupes["nombre_longitudes"] > 1)
    | (analyse_groupes["nombre_latitudes"] > 1)
].index

df_conflits = df[
    df["id_pdc_itinerance"].isin(ids_conflit)
].copy()

print("Nombre d'identifiants conflictuels :", len(ids_conflit))
print("Nombre de lignes conservées dans df_conflits :", len(df_conflits))

Nombre d'identifiants conflictuels : 37115
Nombre de lignes conservées dans df_conflits : 77246


77 246 lignes dans df_conflits = toutes les lignes correspondant à ces 37 115 bornes.(meme bornes on peut avoir deux sources differebtes

# Séparer les identifiants exploitables des identifiants non exploitables


In [25]:
# Séparer les identifiants exploitables des identifiants non exploitables

mask_id_valide = (
    df["id_pdc_itinerance"].notna()
    & (df["id_pdc_itinerance"].str.strip() != "")
    & (df["id_pdc_itinerance"] != "Non concerné")
)

df_ids_valides = df[mask_id_valide].copy()
df_ids_non_valides = df[~mask_id_valide].copy()

print("Lignes avec identifiant valide :", len(df_ids_valides))
print("Lignes avec identifiant non exploitable :", len(df_ids_non_valides))

Lignes avec identifiant valide : 227114
Lignes avec identifiant non exploitable : 118


La logique est donc :

dédoublonner uniquement df_ids_valides ;
garder df_ids_non_valides à part ;
les réintégrer ensuite sans les dédoublonner.

# 1. Trie par versions

In [26]:
df_ids_valides = df_ids_valides.sort_values(
    by=[
        "id_pdc_itinerance",
        "date_maj",
        "last_modified"
    ],
    ascending=True,
    na_position="first"
)
#Garder la version la plus récente
df_ids_dedoublonnes = df_ids_valides.drop_duplicates(
    subset=["id_pdc_itinerance"],
    keep="last"
)
print("Lignes avec identifiants valides avant :", len(df_ids_valides))
print("Lignes avec identifiants valides après :", len(df_ids_dedoublonnes))
print(
    "Lignes retirées :",
    len(df_ids_valides) - len(df_ids_dedoublonnes)
)

Lignes avec identifiants valides avant : 227114
Lignes avec identifiants valides après : 160137
Lignes retirées : 66977


In [27]:
# on reconstitut le dataset 
df = pd.concat(
    [
        df_ids_dedoublonnes,
        df_ids_non_valides
    ],
    ignore_index=True
)

print("Lignes avant :", len(df_brut))
print("Lignes après :", len(df))
print("Lignes retirées :", len(df_brut) - len(df))

Lignes avant : 227232
Lignes après : 160255
Lignes retirées : 66977


In [28]:
mask_id_valide_final = (
    df["id_pdc_itinerance"].notna()
    & (df["id_pdc_itinerance"].str.strip() != "")
    & (df["id_pdc_itinerance"] != "Non concerné")
)

repetitions_restantes = (
    df.loc[
        mask_id_valide_final,
        "id_pdc_itinerance"
    ]
    .duplicated()
    .sum()
)

print("Identifiants valides répétés restants :", repetitions_restantes)

Identifiants valides répétés restants : 0


### Résultat du dédoublonnage

Le dataset contenait initialement **227 232 lignes**.

Après application de la règle de dédoublonnage :

- **160 137 lignes** avec un identifiant valide ont été conservées ;
- **118 lignes** avec un identifiant non exploitable ont été conservées sans dédoublonnage ;
- le dataset final contient **160 255 lignes** ;
- **66 977 lignes** ont été retirées.

La règle appliquée consiste à conserver, pour chaque `id_pdc_itinerance` valide, la version la plus récente selon `date_maj`, puis `last_modified` en cas d’égalité.

Les lignes dont l’identifiant est vide ou égal à `Non concerné` ont été conservées afin d’éviter de supprimer des bornes potentiellement différentes.

In [29]:
# Vérifier les identifiants non exploitables conservés dans df

mask_non_valides_final = (
    df["id_pdc_itinerance"].isna()
    | (df["id_pdc_itinerance"].astype("string").str.strip() == "")
    | (df["id_pdc_itinerance"] == "Non concerné")
)

print(
    "Nombre de lignes non exploitables conservées dans df :",
    mask_non_valides_final.sum()
)

display(
    df.loc[
        mask_non_valides_final,
        ["id_pdc_itinerance", "nom_station", "adresse_station"]
    ].head(20)
)

Nombre de lignes non exploitables conservées dans df : 118


,id_pdc_itinerance,nom_station,adresse_station
160137,Non concerné,Aubade - Comptoir des Fers,"1 Rue des Biches, 74100 Ville-la-Grand"
160138,Non concerné,Parking MAMBHOME,1 Impasse de Recouvrance 17100 SAINTES
160139,Non concerné,"BORNE SECURITEST, DOUVRES LA DELIVRANDE",Rue Jean Perrin 14440 Douvres-la-Délivrande
160140,Non concerné,La jabotte,13 Avenue Max Maurey 06160 Antibes
160141,Non concerné,NANTEUIL,12 Av. Fridingen 77100 Nanteuil-lès-Meaux France
160142,Non concerné,Parking P4 électrique,"2 Rue de Finlande, 69124 Colombier-Saugnieu"
160143,Non concerné,Parking Boulangerie,6 rue edouard branly
160144,Non concerné,Absolut Bougival,21 chemin du haut du trou martin
160145,Non concerné,ALC CAR,2 RUE BRANLY 77400
160146,Non concerné,"Borne, Domaine, Charton-Vachet",25 rue du May


## Conclusion de la partie 2 — Grain et doublons

Le fichier contient plusieurs lignes pour un même `id_pdc_itinerance`, car les données proviennent de plusieurs ressources et livraisons.

La règle retenue consiste à :

- conserver une copie du dataset brut ;
- exclure temporairement les identifiants vides ou égaux à `Non concerné` du dédoublonnage ;
- trier les versions selon `date_maj`, puis `last_modified` ;
- conserver la version la plus récente pour chaque identifiant valide ;
- conserver séparément les groupes présentant des conflits ;
- réintégrer les lignes sans identifiant exploitable.

Après dédoublonnage, le dataset contient 160 255 lignes et aucun identifiant valide ne reste répété.

# Partie 3 — Nettoyage et enrichissement

## Objectif

Cette partie vise à améliorer la qualité du dataset dédoublonné en traitant :

1. les problèmes d’encodage des textes ;
2. les formats des communes et des dates ;
3. les erreurs d’unité dans les puissances ;
4. les codes postaux manquants grâce au géocodage inverse.

Les transformations seront mesurées afin de comparer la qualité avant et après nettoyage.


# 3.1 Conserver l’état avant nettoyage

In [30]:
df_avant_nettoyage = df.copy()
print("Lignes avant nettoyage :", len(df_avant_nettoyage))

Lignes avant nettoyage : 160255


In [31]:
display(
    df["condition_acces"]
    .value_counts(dropna=False)
    .to_frame("nombre")
)

,nombre
condition_acces,
Accès libre,136486
Accès réservé,22560
Accčs libre,1033
Accs libre,171
Acc¸s libre,4
AccĂ¨s libre,1


In [32]:
for valeur, nombre in df["condition_acces"].value_counts(dropna=False).items():
    print(repr(valeur), ":", nombre)

'Accès libre' : 136486
'Accès réservé' : 22560
'Accčs libre' : 1033
'Acc\x8fs libre' : 171
'Acc¸s libre' : 4
'AccĂ¨s libre' : 1


In [33]:
# Sauvegarde avant correction
condition_acces_avant = df["condition_acces"].copy()

# Correction des variantes connues
corrections_condition_acces = {
    "Accčs libre": "Accès libre",
    "Acc\x8fs libre": "Accès libre",
    "Acc¸s libre": "Accès libre",
    "AccĂ¨s libre": "Accès libre"
}

df["condition_acces"] = df["condition_acces"].replace(
    corrections_condition_acces
)

In [34]:
for valeur, nombre in df["condition_acces"].value_counts(dropna=False).items():
    print(repr(valeur), ":", nombre)

'Accès libre' : 137695
'Accès réservé' : 22560


In [35]:
# Normaliser les communes et compter les modifications
commune_avant = df["consolidated_commune"].astype("string")

df["consolidated_commune"] = (
    commune_avant
    .str.strip()
    .str.upper()
)

print(
    "Communes normalisées :",
    (commune_avant != df["consolidated_commune"]).sum()
)

Communes normalisées : 96848


In [36]:
verification_communes = pd.DataFrame({
    "avant": commune_avant,
    "apres": df["consolidated_commune"]
})

verification_communes = verification_communes[
    verification_communes["avant"] != verification_communes["apres"]
]

print("Nombre de communes modifiées :", len(verification_communes))
display(verification_communes.head(15))

Nombre de communes modifiées : 96848


,avant,apres
28,Trets,TRETS
29,Trets,TRETS
30,Bailleul,BAILLEUL
31,Bailleul,BAILLEUL
69,Toulouse,TOULOUSE
70,Toulouse,TOULOUSE
71,Montestruc-sur-Gers,MONTESTRUC-SUR-GERS
72,Montestruc-sur-Gers,MONTESTRUC-SUR-GERS
73,Montestruc-sur-Gers,MONTESTRUC-SUR-GERS
74,Montestruc-sur-Gers,MONTESTRUC-SUR-GERS


In [37]:
import re
import unicodedata
import pandas as pd
from ftfy import fix_text


# ============================================================
# CONFIGURATION
# ============================================================

COLONNE_ADRESSE = "adresse_station"
COLONNE_STATION = "id_station_itinerance"

COLONNE_ORIGINALE = "adresse_station_originale"
COLONNE_METHODE = "adresse_methode_correction"
COLONNE_SCORE_AVANT = "adresse_score_anomalie_avant"
COLONNE_SCORE_APRES = "adresse_score_anomalie_apres"


# ============================================================
# 1. CONSERVATION DE LA VALEUR D'ORIGINE
# ============================================================

if COLONNE_ORIGINALE not in df.columns:
    df[COLONNE_ORIGINALE] = df[COLONNE_ADRESSE].copy()


# ============================================================
# 2. RÈGLES DE DÉTECTION
# ============================================================

ACCENTS_FRANCAIS = set(
    "ÀÂÄÆÇÉÈÊËÎÏÔÖŒÙÛÜŸ"
    "àâäæçéèêëîïôöœùûüÿ"
)

REGEX_MOJIBAKE = re.compile(
    r"Ã.|Â.|â€.|â€™|â€œ|â€|ðŸ.|√.|ï»¿"
)


def normaliser_texte(texte):
    """Normalise Unicode et supprime les espaces inutiles."""

    texte = unicodedata.normalize("NFC", str(texte))
    texte = texte.replace("\u00A0", " ")
    texte = re.sub(r"\s+", " ", texte)

    return texte.strip()


def score_encodage(texte):
    """
    Calcule un score d'anomalie.
    Plus le score est élevé, plus le texte est suspect.
    """

    if pd.isna(texte):
        return 0

    texte = normaliser_texte(texte)
    score = 0

    # Séquences classiques de mauvais encodage
    score += 8 * len(REGEX_MOJIBAKE.findall(texte))

    for caractere in texte:
        code = ord(caractere)
        categorie = unicodedata.category(caractere)
        nom_unicode = unicodedata.name(caractere, "")

        # Caractère de remplacement : information perdue
        if caractere == "�":
            score += 20

        # Caractères de contrôle C1
        elif 0x80 <= code <= 0x9F:
            score += 12

        # Autres caractères de contrôle
        elif categorie.startswith("C") and caractere not in "\t\n\r":
            score += 8

        # Lettres latines inhabituelles dans une adresse française
        elif (
            caractere.isalpha()
            and "LATIN" in nom_unicode
            and code > 255
            and caractere not in ACCENTS_FRANCAIS
        ):
            score += 4

    return score


# ============================================================
# 3. RÉPARATION DES CARACTÈRES DE CONTRÔLE
# ============================================================

def remplacer_controles_c1(texte, encodage):
    """Tente de décoder les caractères C1 avec un ancien encodage."""

    resultat = []

    for caractere in texte:
        code = ord(caractere)

        if 0x80 <= code <= 0x9F:
            try:
                resultat.append(bytes([code]).decode(encodage))
            except (UnicodeDecodeError, LookupError):
                resultat.append(caractere)
        else:
            resultat.append(caractere)

    return "".join(resultat)


# ============================================================
# 4. CORRECTION AUTOMATIQUE D'UNE ADRESSE
# ============================================================

def reparer_adresse(valeur):
    """
    Teste plusieurs réparations et conserve uniquement
    celle qui diminue le score d'anomalie.
    """

    if pd.isna(valeur):
        return pd.NA, "manquante", 0, 0

    texte_original = normaliser_texte(valeur)
    score_avant = score_encodage(texte_original)

    # Le texte est déjà propre
    if score_avant == 0:
        return texte_original, "aucune", 0, 0

    candidats = [
        ("original", texte_original),
        ("ftfy", normaliser_texte(fix_text(texte_original)))
    ]

    # Encodages couramment rencontrés dans les anciennes données
    for encodage in ["mac_roman", "cp1252", "cp850"]:

        candidat = remplacer_controles_c1(
            texte_original,
            encodage
        )

        candidat = normaliser_texte(candidat)

        candidats.append(
            (f"controle_c1_{encodage}", candidat)
        )

        candidats.append(
            (
                f"controle_c1_{encodage}_ftfy",
                normaliser_texte(fix_text(candidat))
            )
        )

    # Suppression des candidats identiques
    candidats_uniques = []
    textes_deja_testes = set()

    for methode, candidat in candidats:
        if candidat not in textes_deja_testes:
            candidats_uniques.append((methode, candidat))
            textes_deja_testes.add(candidat)

    # Sélection du candidat avec le score le plus faible
    meilleure_methode, meilleur_texte = min(
        candidats_uniques,
        key=lambda resultat: score_encodage(resultat[1])
    )

    score_apres = score_encodage(meilleur_texte)

    # La correction est appliquée uniquement si elle améliore le texte
    if score_apres < score_avant:
        return (
            meilleur_texte,
            meilleure_methode,
            score_avant,
            score_apres
        )

    return (
        texte_original,
        "non_resolu",
        score_avant,
        score_avant
    )


# ============================================================
# 5. PREMIÈRE PASSE : FTFY ET ANCIENS ENCODAGES
# ============================================================

resultats = df[COLONNE_ORIGINALE].apply(reparer_adresse)

df[
    [
        COLONNE_ADRESSE,
        COLONNE_METHODE,
        COLONNE_SCORE_AVANT,
        COLONNE_SCORE_APRES
    ]
] = pd.DataFrame(
    resultats.tolist(),
    index=df.index
)


# ============================================================
# 6. DEUXIÈME PASSE : RÉCUPÉRATION PAR STATION
# ============================================================

nombre_recuperees_station = 0

if COLONNE_STATION in df.columns:

    # Adresses fiables disponibles pour chaque station
    masque_reference = (
        df[COLONNE_STATION].notna()
        & df[COLONNE_ADRESSE].notna()
        & (df[COLONNE_SCORE_APRES] == 0)
    )

    # Adresse propre la plus fréquente par station
    adresses_references = (
        df.loc[masque_reference]
        .groupby(COLONNE_STATION)[COLONNE_ADRESSE]
        .agg(lambda valeurs: valeurs.value_counts().index[0])
    )

    # Lignes encore non résolues
    masque_non_resolu = (
        df[COLONNE_STATION].notna()
        & (df[COLONNE_SCORE_APRES] > 0)
    )

    adresses_recuperees = (
        df.loc[masque_non_resolu, COLONNE_STATION]
        .map(adresses_references)
    )

    index_recuperes = adresses_recuperees.dropna().index
    nombre_recuperees_station = len(index_recuperes)

    # Remplacement uniquement si une adresse propre existe
    df.loc[index_recuperes, COLONNE_ADRESSE] = (
        adresses_recuperees.loc[index_recuperes]
    )

    df.loc[
        index_recuperes,
        COLONNE_METHODE
    ] = "recuperee_meme_station"

    # Recalcul du score
    df.loc[
        index_recuperes,
        COLONNE_SCORE_APRES
    ] = df.loc[
        index_recuperes,
        COLONNE_ADRESSE
    ].apply(score_encodage)


# ============================================================
# 7. TRAÇABILITÉ ET STATUT FINAL
# ============================================================

df["adresse_corrigee_encodage"] = (
    df[COLONNE_ORIGINALE].fillna("")
    != df[COLONNE_ADRESSE].fillna("")
)

df["adresse_encodage_suspect"] = (
    df[COLONNE_SCORE_APRES] > 0
)

df["statut_adresse"] = "valide"

df.loc[
    df["adresse_corrigee_encodage"],
    "statut_adresse"
] = "corrigee_automatiquement"

df.loc[
    df[COLONNE_METHODE] == "recuperee_meme_station",
    "statut_adresse"
] = "recuperee_meme_station"

df.loc[
    df["adresse_encodage_suspect"],
    "statut_adresse"
] = "a_verifier"

df.loc[
    df[COLONNE_ADRESSE].isna(),
    "statut_adresse"
] = "manquante"


# ============================================================
# 8. CONTRÔLES
# ============================================================

nombre_corrigees = int(
    df["adresse_corrigee_encodage"].sum()
)

nombre_suspectes = int(
    df["adresse_encodage_suspect"].sum()
)

nombre_degradees = int(
    (
        df[COLONNE_SCORE_APRES]
        > df[COLONNE_SCORE_AVANT]
    ).sum()
)

print("Adresses corrigées :", nombre_corrigees)
print(
    "Adresses récupérées depuis la même station :",
    nombre_recuperees_station
)
print("Adresses encore suspectes :", nombre_suspectes)
print("Adresses dégradées par le traitement :", nombre_degradees)
print("Adresses manquantes :", int(df[COLONNE_ADRESSE].isna().sum()))

assert nombre_degradees == 0, (
    "Certaines adresses ont été dégradées par le traitement."
)


# ============================================================
# 9. BILAN DES STATUTS
# ============================================================

display(
    df["statut_adresse"]
    .value_counts(dropna=False)
    .rename_axis("statut")
    .reset_index(name="nombre")
)


# ============================================================
# 10. EXEMPLES AVANT / APRÈS
# ============================================================

display(
    df.loc[
        df["adresse_corrigee_encodage"],
        [
            COLONNE_ORIGINALE,
            COLONNE_ADRESSE,
            COLONNE_METHODE,
            COLONNE_SCORE_AVANT,
            COLONNE_SCORE_APRES,
            "statut_adresse"
        ]
    ]
    .drop_duplicates()
    .head(30)
)


# ============================================================
# 11. CAS ENCORE NON RÉSOLUS
# ============================================================

display(
    df.loc[
        df["adresse_encodage_suspect"],
        [
            COLONNE_ORIGINALE,
            COLONNE_ADRESSE,
            COLONNE_METHODE,
            COLONNE_SCORE_APRES,
            "statut_adresse"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        COLONNE_SCORE_APRES,
        ascending=False
    )
    .head(10)
)

Adresses corrigées : 32488
Adresses récupérées depuis la même station : 9
Adresses encore suspectes : 359
Adresses dégradées par le traitement : 0
Adresses manquantes : 0


,statut,nombre
0,valide,127481
1,corrigee_automatiquement,32406
2,a_verifier,359
3,recuperee_meme_station,9


,adresse_station_originale,adresse_station,adresse_methode_correction,adresse_score_anomalie_avant,adresse_score_anomalie_apres,statut_adresse
100,208 Chemin du Rieu 30190 Sainte-Anastasie,208 Chemin du Rieu 30190 Sainte-Anastasie,aucune,0,0,corrigee_automatiquement
109,"D924, Écouché-les-Vallées 61200 France","D924, Écouché-les-Vallées 61200 France",aucune,0,0,corrigee_automatiquement
125,"N10, Aussac-Vadalle 16560 France","N10, Aussac-Vadalle 16560 France",aucune,0,0,corrigee_automatiquement
129,"A71/E11, km200, Aire de Bourges-Marmagne, Mar...","A71/E11, km200, Aire de Bourges-Marmagne, Marm...",aucune,0,0,corrigee_automatiquement
160,"D53, La Madeleine-de-Nonancourt 27320 France","D53, La Madeleine-de-Nonancourt 27320 France",aucune,0,0,corrigee_automatiquement
164,"Autoroute A28 (sortie 14), Courbépine 27300 F...","Autoroute A28 (sortie 14), Courbépine 27300 Fr...",aucune,0,0,corrigee_automatiquement
181,"A85, km115, Athée-sur-Cher 37270 France","A85, km115, Athée-sur-Cher 37270 France",aucune,0,0,corrigee_automatiquement
220,"A63, Saugnac-et-Muret 40410 France","A63, Saugnac-et-Muret 40410 France",aucune,0,0,corrigee_automatiquement
266,"A63, Lesperon 40260 France","A63, Lesperon 40260 France",aucune,0,0,corrigee_automatiquement
311,"A19 (sortie 6), Beaune-la-Rolande 45340 France","A19 (sortie 6), Beaune-la-Rolande 45340 France",aucune,0,0,corrigee_automatiquement


,adresse_station_originale,adresse_station,adresse_methode_correction,adresse_score_anomalie_apres,statut_adresse
48278,Chemin B� Rivi� 97126 Deshaies,Chemin B� Rivi� 97126 Deshaies,non_resolu,40,a_verifier
48302,Rue D�bidine Saha� La Jaille 97122 Baie-Mahault,Rue D�bidine Saha� La Jaille 97122 Baie-Mahault,non_resolu,40,a_verifier
48370,Port de p�che 97118 Saint-Fran�ois,Port de p�che 97118 Saint-Fran�ois,non_resolu,40,a_verifier
48332,314 Chemin de l'Embarcad�re 97114 Trois-Rivi�res,314 Chemin de l'Embarcad�re 97114 Trois-Rivi�res,non_resolu,40,a_verifier
48300,Rue D�bidine Saha� Fond Sahai 97122 Baie-Mahault,Rue D�bidine Saha� Fond Sahai 97122 Baie-Mahault,non_resolu,40,a_verifier
48364,"Avenue de l'Europe, 97118 Saint-Fran�ois","Avenue de l'Europe, 97118 Saint-Fran�ois",non_resolu,20,a_verifier
48350,Section Cayenne 97118 Saint-Fran�ois,Section Cayenne 97118 Saint-Fran�ois,non_resolu,20,a_verifier
48294,Bd de la R�conciliation,Bd de la R�conciliation,non_resolu,20,a_verifier
48293,1666 Route de la L�zarde 97129 Lamentin,1666 Route de la L�zarde 97129 Lamentin,non_resolu,20,a_verifier
48330,Chemin de Dugommier 97114 Trois-Rivi�re,Chemin de Dugommier 97114 Trois-Rivi�re,non_resolu,20,a_verifier


In [38]:
# Conversion et contrôle des dates
colonnes_dates = [
    "date_maj", "date_mise_en_service",
    "last_modified", "created_at", "updated_at"
]

date_reference = pd.Timestamp("2026-06-26", tz="UTC")
controle_dates = []

for col in [c for c in colonnes_dates if c in df.columns]:
    valeurs_avant = df[col].copy()
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

    controle_dates.append({
        "colonne": col,
        "dates_non_convertibles": max(
            0, int(df[col].isna().sum() - valeurs_avant.isna().sum())
        ),
        "dates_futures": int((df[col] > date_reference).sum())
    })

controle_dates = pd.DataFrame(controle_dates)
display(controle_dates)

,colonne,dates_non_convertibles,dates_futures
0,date_maj,0,56
1,date_mise_en_service,0,0
2,last_modified,0,0
3,created_at,0,0


In [39]:
# Conserver, signaler et neutraliser les dates futures
if "date_maj_originale" not in df.columns:
    df["date_maj_originale"] = df["date_maj"]

date_reference = pd.Timestamp("2026-06-26", tz="UTC")
df["date_maj_future"] = df["date_maj"] > date_reference

df.loc[df["date_maj_future"], "date_maj"] = pd.NaT

print("Dates futures corrigées :", df["date_maj_future"].sum())
print("Dates futures restantes :", (df["date_maj"] > date_reference).sum())

Dates futures corrigées : 56
Dates futures restantes : 0


In [40]:
# Contrôle de la puissance nominale
df["puissance_nominale"] = pd.to_numeric(
    df["puissance_nominale"],
    errors="coerce"
)

puissances_suspectes = df[
    (df["puissance_nominale"] <= 0) |
    (df["puissance_nominale"] > 1000)
]

print("Puissances <= 0 :", (df["puissance_nominale"] <= 0).sum())
print("Puissances > 1000 :", (df["puissance_nominale"] > 1000).sum())

display(
    puissances_suspectes[
        ["id_pdc_itinerance", "puissance_nominale"]
    ].sort_values("puissance_nominale", ascending=False).head(20)
)

Puissances <= 0 : 5211
Puissances > 1000 : 890


,id_pdc_itinerance,puissance_nominale
73630,FRLMSE12348882602,160000.0
159879,FRZWOEZW065352,50000.0
159877,FRZWOEZW065342,50000.0
73629,FRLMSE12348882601,22000.0
157822,FRYAWE12348400332,16000.0
157826,FRYAWE12348400342,16000.0
157821,FRYAWE12348400331,12800.0
157825,FRYAWE12348400341,12800.0
159333,FRZWOEZW008732,7360.0
160115,FRZWOEZW126016,7360.0


In [41]:
# Conversion de la puissance en valeur numérique
df["puissance_nominale"] = pd.to_numeric(
    df["puissance_nominale"],
    errors="coerce"
)

# Conservation de la valeur initiale
if "puissance_nominale_originale" not in df.columns:
    df["puissance_nominale_originale"] = df["puissance_nominale"].copy()

# Détection à partir de la valeur originale
masque_watts = df["puissance_nominale_originale"] > 1000

# Indicateur de traçabilité
df["puissance_corrigee_w_kw"] = masque_watts

# Reconstruction depuis la valeur originale pour rendre la cellule réexécutable
df["puissance_nominale"] = df["puissance_nominale_originale"]

# Correction W → kW
df.loc[masque_watts, "puissance_nominale"] = (
    df.loc[masque_watts, "puissance_nominale_originale"] / 1000
)

print("Puissances corrigées :", int(masque_watts.sum()))
print(
    "Puissances > 1000 restantes :",
    int((df["puissance_nominale"] > 1000).sum())
)

display(
    df.loc[
        masque_watts,
        [
            "puissance_nominale_originale",
            "puissance_nominale",
            "puissance_corrigee_w_kw"
        ]
    ].head(15)
)

Puissances corrigées : 890
Puissances > 1000 restantes : 0


,puissance_nominale_originale,puissance_nominale,puissance_corrigee_w_kw
17093,1242.0,1.242,True
17094,1242.0,1.242,True
17121,1242.0,1.242,True
17122,1242.0,1.242,True
73629,22000.0,22.000,True
73630,160000.0,160.000,True
157821,12800.0,12.800,True
157822,16000.0,16.000,True
157825,12800.0,12.800,True
157826,16000.0,16.000,True


In [42]:
# Analyse des puissances après conversion W → kW
puissances_converties = df.loc[
    df["puissance_corrigee_w_kw"],
    [
        "puissance_nominale_originale",
        "puissance_nominale"
    ]
].copy()

print("Statistiques des puissances converties :")
display(puissances_converties["puissance_nominale"].describe())

print("\nValeurs converties les plus fréquentes :")
display(
    puissances_converties["puissance_nominale"]
    .value_counts()
    .head(20)
    .rename_axis("puissance_kw")
    .reset_index(name="nombre")
)

# Repérage des puissances converties très faibles
masque_conversion_a_verifier = (
    df["puissance_corrigee_w_kw"]
    & (df["puissance_nominale"] < 2)
)

df["puissance_conversion_a_verifier"] = masque_conversion_a_verifier

print(
    "\nConversions inférieures à 2 kW à vérifier :",
    int(masque_conversion_a_verifier.sum())
)

display(
    df.loc[
        masque_conversion_a_verifier,
        [
            "id_pdc_itinerance",
            "nom_station",
            "puissance_nominale_originale",
            "puissance_nominale"
        ]
    ].head(20)
)

Statistiques des puissances converties :


count    890.000000
mean       7.181649
std        5.727963
min        1.200000
25%        7.360000
50%        7.360000
75%        7.360000
max      160.000000
Name: puissance_nominale, dtype: float64


Valeurs converties les plus fréquentes :


,puissance_kw,nombre
0,7.360,769
1,3.680,90
2,1.200,8
3,6.900,4
4,1.840,4
5,1.242,4
6,50.000,2
7,16.000,2
8,12.800,2
9,22.000,1



Conversions inférieures à 2 kW à vérifier : 16


,id_pdc_itinerance,nom_station,puissance_nominale_originale,puissance_nominale
17093,FRCG0E001133,ODICEE - AIX,1242.0,1.242
17094,FRCG0E001134,ODICEE - AIX,1242.0,1.242
17121,FRCG0E001161,ODICEE - AIX,1242.0,1.242
17122,FRCG0E001162,ODICEE - AIX,1242.0,1.242
157829,FRYAWE12349013781,Breteuil,1200.0,1.200
157830,FRYAWE12349013782,Breteuil,1200.0,1.200
157831,FRYAWE12349013783,Breteuil,1200.0,1.200
157832,FRYAWE12349013784,Breteuil,1200.0,1.200
157837,FRYAWE12349013791,Breteuil,1200.0,1.200
157838,FRYAWE12349013792,Breteuil,1200.0,1.200


In [43]:
import numpy as np

df["statut_puissance"] = np.select(
    [
        df["puissance_nominale"].isna(),
        df["puissance_nominale"] <= 0,
        (
            df["puissance_corrigee_w_kw"]
            & (df["puissance_nominale"] < 2)
        ),
        df["puissance_corrigee_w_kw"]
    ],
    [
        "manquante",
        "non_positive",
        "corrigee_a_verifier",
        "corrigee_w_kw"
    ],
    default="valeur_initiale"
)

display(
    df["statut_puissance"]
    .value_counts(dropna=False)
    .rename_axis("statut")
    .reset_index(name="nombre")
)

,statut,nombre
0,valeur_initiale,154154
1,non_positive,5211
2,corrigee_w_kw,874
3,corrigee_a_verifier,16


### Validation de la correction W → kW

La correction a concerné 890 lignes. Après division par 1 000,
aucune puissance supérieure à 1 000 kW ne subsiste.

La majorité des valeurs corrigées devient égale à 7,36 kW ou
3,68 kW. Ces deux valeurs représentent environ 96,5 % des
corrections, ce qui renforce l’hypothèse d’une confusion entre
watts et kilowatts dans les données sources.

Seize valeurs corrigées sont inférieures à 2 kW. Elles apparaissent
de manière répétée au sein des mêmes stations et ne sont donc pas
considérées automatiquement comme erronées. Elles sont conservées
et signalées avec le statut `corrigee_a_verifier`.

Aucune valeur n’a été supprimée ou imputée par une statistique,
afin de préserver la traçabilité et l’information source.

In [44]:
# Détection des puissances nulles ou négatives
masque_puissance_non_positive = df["puissance_nominale"] <= 0

print(
    "Puissances inférieures ou égales à 0 :",
    int(masque_puissance_non_positive.sum())
)

print(
    "Puissances égales à 0 :",
    int((df["puissance_nominale"] == 0).sum())
)

print(
    "Puissances négatives :",
    int((df["puissance_nominale"] < 0).sum())
)

# Répartition des valeurs concernées
display(
    df.loc[
        masque_puissance_non_positive,
        "puissance_nominale"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("puissance_nominale")
    .reset_index(name="nombre")
)

# Aperçu des lignes concernées
colonnes_affichage = [
    colonne for colonne in [
        "id_pdc_itinerance",
        "id_station_itinerance",
        "nom_station",
        "nom_operateur",
        "puissance_nominale_originale",
        "puissance_nominale"
    ]
    if colonne in df.columns
]

display(
    df.loc[
        masque_puissance_non_positive,
        colonnes_affichage
    ].head(20)
)

Puissances inférieures ou égales à 0 : 5211
Puissances égales à 0 : 5211
Puissances négatives : 0


,puissance_nominale,nombre
0,0.0,5211


,id_pdc_itinerance,id_station_itinerance,nom_station,nom_operateur,puissance_nominale_originale,puissance_nominale
3021,FR911EFR00020000000010000001,FR911EFR00020000000010000001,Porsche Sales & Marketplace Centre Porsche Lil...,Porsche Sales & Marketplace,0.0,0.0
3022,FR911EFR00020000000020000001,FR911EFR00020000000020000001,Porsche Sales & Marketplace Centre Porsche Lil...,Porsche Sales & Marketplace,0.0,0.0
3023,FR911EFR00025500000010000001,FR911EFR00025500000010000001,Porsche Sales & Marketplace Centre Porsche Fré...,Porsche Sales & Marketplace,0.0,0.0
3024,FR911EFR00025500000020000001,FR911EFR00025500000020000001,Porsche Sales & Marketplace Centre Porsche Fré...,Porsche Sales & Marketplace,0.0,0.0
3025,FR911EFR00026300000010000001,FR911EFR00026300000010000001,Porsche Sales & Marketplace Centre Porsche Vél...,Porsche Sales & Marketplace,0.0,0.0
3026,FR911EFR00026300000020000001,FR911EFR00026300000020000001,Porsche Sales & Marketplace Centre Porsche Vél...,Porsche Sales & Marketplace,0.0,0.0
3027,FR911EFR00026400000010000001,FR911EFR00026400000010000001,Porsche Sales & Marketplace Centre Service Por...,Porsche Sales & Marketplace,0.0,0.0
3028,FR911EFR00026400000020000001,FR911EFR00026400000020000001,Porsche Sales & Marketplace Centre Service Por...,Porsche Sales & Marketplace,0.0,0.0
3029,FR911EFR00026500000010000001,FR911EFR00026500000010000001,Porsche Sales & Marketplace Centre Porsche Tou...,Porsche Sales & Marketplace,0.0,0.0
3030,FR911EFR00026500000020000001,FR911EFR00026500000020000001,Porsche Sales & Marketplace Centre Porsche Tou...,Porsche Sales & Marketplace,0.0,0.0


In [45]:
import numpy as np

# Conservation de la valeur initiale
if "puissance_nominale_originale" not in df.columns:
    df["puissance_nominale_originale"] = df["puissance_nominale"].copy()

# Détection des valeurs non valides
masque_puissance_non_positive = df["puissance_nominale"] <= 0

# Création d'un indicateur de qualité
df["puissance_non_positive"] = masque_puissance_non_positive

# Neutralisation : valeur invalide remplacée par NaN
df.loc[
    masque_puissance_non_positive,
    "puissance_nominale"
] = np.nan

print(
    "Puissances non positives neutralisées :",
    int(masque_puissance_non_positive.sum())
)

print(
    "Puissances inférieures ou égales à 0 restantes :",
    int((df["puissance_nominale"] <= 0).sum())
)

print(
    "Valeurs manquantes dans puissance_nominale après traitement :",
    int(df["puissance_nominale"].isna().sum())
)

Puissances non positives neutralisées : 5211
Puissances inférieures ou égales à 0 restantes : 0
Valeurs manquantes dans puissance_nominale après traitement : 5211


### Traitement des puissances nulles ou négatives

L'analyse a identifié 5 211 puissances nominales inférieures ou égales
à zéro.

Ces valeurs sont considérées comme invalides, car une borne de recharge
en service ne peut pas avoir une puissance nominale nulle ou négative.

Elles n'ont pas été remplacées par la moyenne ou la médiane, car la
puissance dépend des caractéristiques techniques de chaque équipement.
Une imputation statistique pourrait donc créer des valeurs artificielles.

Les valeurs initiales sont conservées dans
`puissance_nominale_originale`. Les anomalies sont signalées par la
colonne `puissance_non_positive`, puis neutralisées avec une valeur
manquante `NaN`.

Les lignes correspondantes sont conservées afin de ne pas perdre les
autres informations relatives aux bornes.

###  Mesure des codes postaux manquants

Avant de procéder à l’enrichissement, nous mesurons :

- le nombre de codes postaux absents ;
- le pourcentage de valeurs manquantes ;
- le nombre de lignes possédant des coordonnées exploitables.

Cette mesure constituera le point de comparaison avant/après.

In [46]:
colonne_cp = "consolidated_code_postal"
colonne_lon = "consolidated_longitude"
colonne_lat = "consolidated_latitude"

# Une chaîne vide est considérée comme une valeur manquante
df[colonne_cp] = (
    df[colonne_cp]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })
)

nombre_lignes = len(df)
nombre_cp_manquants_avant = df[colonne_cp].isna().sum()
taux_cp_manquants_avant = (
    nombre_cp_manquants_avant / nombre_lignes * 100
)

print(f"Nombre total de lignes : {nombre_lignes:,}")
print(
    f"Codes postaux manquants avant enrichissement : "
    f"{nombre_cp_manquants_avant:,}"
)
print(
    f"Taux de codes postaux manquants : "
    f"{taux_cp_manquants_avant:.2f} %"
)

Nombre total de lignes : 160,255
Codes postaux manquants avant enrichissement : 75,548
Taux de codes postaux manquants : 47.14 %


###  Validation préalable des coordonnées GPS

Le géocodage inverse est limité aux lignes dont les coordonnées sont
suffisamment fiables.

Les contrôles appliqués sont les suivants :

- longitude et latitude présentes ;
- valeurs convertibles en nombres ;
- longitude comprise entre -180 et 180 ;
- latitude comprise entre -90 et 90 ;
- exclusion du point `(0, 0)` ;
- présence dans une zone géographique compatible avec la France.

Le rectangle géographique utilisé constitue un préfiltre. Il permet d’écarter
les coordonnées manifestement aberrantes, mais il ne garantit pas à lui seul
qu’un point se trouve sur la terre ferme.

In [47]:
# Conservation des coordonnées originales
if "longitude_originale" not in df.columns:
    df["longitude_originale"] = df[colonne_lon].copy()

if "latitude_originale" not in df.columns:
    df["latitude_originale"] = df[colonne_lat].copy()

# Conversion en valeurs numériques
df[colonne_lon] = pd.to_numeric(
    df[colonne_lon],
    errors="coerce"
)

df[colonne_lat] = pd.to_numeric(
    df[colonne_lat],
    errors="coerce"
)

# Contrôle des limites mondiales
coordonnees_numeriques_valides = (
    df[colonne_lon].between(-180, 180)
    & df[colonne_lat].between(-90, 90)
)

# Exclusion du point (0, 0)
coordonnees_non_nulles = ~(
    df[colonne_lon].eq(0)
    & df[colonne_lat].eq(0)
)

# Rectangle englobant approximatif de la France métropolitaine et de la Corse
coordonnees_dans_zone_france = (
    df[colonne_lon].between(-5.6, 9.9)
    & df[colonne_lat].between(41.0, 51.7)
)

df["gps_valide_pour_geocodage"] = (
    coordonnees_numeriques_valides
    & coordonnees_non_nulles
    & coordonnees_dans_zone_france
)

bilan_gps = pd.DataFrame({
    "Indicateur": [
        "Coordonnées numériques valides",
        "Coordonnées dans la zone France",
        "Coordonnées retenues pour géocodage",
        "Coordonnées non retenues"
    ],
    "Nombre de lignes": [
        int(coordonnees_numeriques_valides.sum()),
        int(coordonnees_dans_zone_france.sum()),
        int(df["gps_valide_pour_geocodage"].sum()),
        int((~df["gps_valide_pour_geocodage"]).sum())
    ]
})

display(bilan_gps)

,Indicateur,Nombre de lignes
0,Coordonnées numériques valides,160255
1,Coordonnées dans la zone France,159502
2,Coordonnées retenues pour géocodage,159502
3,Coordonnées non retenues,753


### 3. Comparaison avec l’indicateur de qualité fourni par la source

La colonne `consolidated_is_lon_lat_correct` indique si les coordonnées sont
considérées comme correctes dans le fichier consolidé.

Cet indicateur est utilisé comme information complémentaire.

Il n’est pas utilisé seul, car une coordonnée peut être située dans les limites
géographiques attendues tout en restant imprécise ou mal positionnée.

In [48]:
df["flag_gps_source"] = (
    df["consolidated_is_lon_lat_correct"]
    .fillna(False)
    .astype(bool)
)

df["gps_conflit_controle_source"] = (
    df["gps_valide_pour_geocodage"]
    != df["flag_gps_source"]
)

print(
    "Nombre de désaccords entre notre contrôle et le flag source :",
    f"{df['gps_conflit_controle_source'].sum():,}"
)

display(
    df.loc[
        df["gps_conflit_controle_source"],
        [
            "id_pdc_itinerance",
            "nom_station",
            colonne_lon,
            colonne_lat,
            "flag_gps_source",
            "gps_valide_pour_geocodage"
        ]
    ].head(10)
)

Nombre de désaccords entre notre contrôle et le flag source : 66,184


,id_pdc_itinerance,nom_station,consolidated_longitude,consolidated_latitude,flag_gps_source,gps_valide_pour_geocodage
0,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,7.762694,48.825613,False,True
1,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,7.762694,48.825613,False,True
2,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,7.762694,48.825613,False,True
3,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,7.762694,48.825613,False,True
4,ATHTBE1004957,HAG_P22_Slave_3,7.761841,48.827040,False,True
5,ATHTBE1004958,HAG_P22_Slave_3,7.761841,48.827040,False,True
6,ATHTBE1004959,HAG_P22_Slave_3,7.761841,48.827040,False,True
7,ATHTBE1004960,HAG_P22_Slave_3,7.761841,48.827040,False,True
8,ATHTBE1004961,HAG_P21_Slave_4,7.763987,48.826323,False,True
9,ATHTBE1004962,HAG_P21_Slave_4,7.763987,48.826323,False,True


## Géocodage inverse des codes postaux manquants

Les codes postaux manquants sont complétés à partir des coordonnées GPS.

Seules les lignes ayant :

- un code postal manquant ;
- une longitude renseignée ;
- une latitude renseignée ;
- des coordonnées jugées valides ;

sont utilisées pour le géocodage inverse.

In [49]:
echantillon_geocodage = df.loc[
    df["consolidated_code_postal"].isna()
    & df["gps_valide_pour_geocodage"],
    [
        "consolidated_longitude",
        "consolidated_latitude"
    ]
].drop_duplicates().head(20)

display(echantillon_geocodage)

,consolidated_longitude,consolidated_latitude
0,7.762694,48.825613
4,7.761841,48.827040
8,7.763987,48.826323
13,7.763863,48.826358
20,7.761358,48.827185
21,7.761165,48.827255
25,2.367618,49.877237
32,7.763606,48.826447
35,4.764598,50.240043
37,6.017766,50.639134


In [50]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

geolocator = Nominatim(
    user_agent="tp_irve_geocodage"
)

reverse = RateLimiter(
    geolocator.reverse,
    min_delay_seconds=1
)

def recuperer_code_postal(ligne):

    resultat = reverse(
        (
            ligne["consolidated_latitude"],
            ligne["consolidated_longitude"]
        ),
        language="fr",
        addressdetails=True
    )

    if resultat is None:
        return pd.Series({
            "code_postal_geocode": pd.NA,
            "commune_geocode": pd.NA
        })

    adresse = resultat.raw.get("address", {})

    commune = (
        adresse.get("city")
        or adresse.get("town")
        or adresse.get("village")
        or adresse.get("municipality")
    )

    return pd.Series({
        "code_postal_geocode": adresse.get("postcode", pd.NA),
        "commune_geocode": commune
    })

In [51]:
resultats_geocodage = echantillon_geocodage.apply(
    recuperer_code_postal,
    axis=1
)

echantillon_geocodage = pd.concat(
    [
        echantillon_geocodage.reset_index(drop=True),
        resultats_geocodage.reset_index(drop=True)
    ],
    axis=1
)

display(echantillon_geocodage)

,consolidated_longitude,consolidated_latitude,code_postal_geocode,commune_geocode
0,7.762694,48.825613,67500,Haguenau
1,7.761841,48.827040,67500,Haguenau
2,7.763987,48.826323,67500,Haguenau
3,7.763863,48.826358,67500,Haguenau
4,7.761358,48.827185,67500,Haguenau
5,7.761165,48.827255,67500,Haguenau
6,2.367618,49.877237,80330,Longueau
7,7.763606,48.826447,67500,Haguenau
8,4.764598,50.240043,5520,Anthée
9,6.017766,50.639134,4700,Eupen


### Résultat

Le géocodage inverse permet de récupérer un code postal et une commune à partir
des coordonnées GPS.

La méthode a été testée sur un échantillon de coordonnées valides et fournit des
résultats cohérents.

Le traitement complet pourrait être appliqué à l'ensemble des coordonnées
uniques, mais il n'est pas exécuté ici afin de limiter le temps de calcul et les
appels au service externe.

In [52]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

geolocator = Nominatim(user_agent="tp_irve_geocodage")

reverse = RateLimiter(
    geolocator.reverse,
    min_delay_seconds=1
)

def recuperer_code_postal(ligne):
    resultat = reverse(
        (
            ligne["consolidated_latitude"],
            ligne["consolidated_longitude"]
        ),
        language="fr",
        addressdetails=True
    )

    if resultat is None:
        return pd.Series({
            "code_postal_geocode": pd.NA,
            "commune_geocode": pd.NA
        })

    adresse = resultat.raw.get("address", {})

    commune = (
        adresse.get("city")
        or adresse.get("town")
        or adresse.get("village")
        or adresse.get("municipality")
    )

    return pd.Series({
        "code_postal_geocode": adresse.get("postcode", pd.NA),
        "commune_geocode": commune
    })


resultats_geocodage = echantillon_geocodage.apply(
    recuperer_code_postal,
    axis=1
)

echantillon_geocodage = pd.concat(
    [
        echantillon_geocodage.reset_index(drop=True),
        resultats_geocodage.reset_index(drop=True)
    ],
    axis=1
)

display(echantillon_geocodage)

,consolidated_longitude,consolidated_latitude,code_postal_geocode,commune_geocode,code_postal_geocode,commune_geocode
0,7.762694,48.825613,67500,Haguenau,67500,Haguenau
1,7.761841,48.827040,67500,Haguenau,67500,Haguenau
2,7.763987,48.826323,67500,Haguenau,67500,Haguenau
3,7.763863,48.826358,67500,Haguenau,67500,Haguenau
4,7.761358,48.827185,67500,Haguenau,67500,Haguenau
5,7.761165,48.827255,67500,Haguenau,67500,Haguenau
6,2.367618,49.877237,80330,Longueau,80330,Longueau
7,7.763606,48.826447,67500,Haguenau,67500,Haguenau
8,4.764598,50.240043,5520,Anthée,5520,Anthée
9,6.017766,50.639134,4700,Eupen,4700,Eupen


In [53]:
print(
    "Codes postaux récupérés :",
    echantillon_geocodage["code_postal_geocode"].notna().sum(),
    "/",
    len(echantillon_geocodage)
)

Codes postaux récupérés : code_postal_geocode    20
code_postal_geocode    20
dtype: int64 / 20


In [54]:
echantillon_geocodage = echantillon_geocodage.loc[
    :,
    ~echantillon_geocodage.columns.duplicated()
].copy()

print(echantillon_geocodage.columns.tolist())

['consolidated_longitude', 'consolidated_latitude', 'code_postal_geocode', 'commune_geocode']


In [55]:
nombre_codes_recuperes = (
    echantillon_geocodage["code_postal_geocode"]
    .notna()
    .sum()
)

print(
    f"Codes postaux récupérés : "
    f"{nombre_codes_recuperes} / {len(echantillon_geocodage)}"
)

Codes postaux récupérés : 20 / 20


## Détection et correction des unités de puissance

Certaines valeurs de puissance peuvent avoir été renseignées en watts au lieu
de kilowatts.

Par exemple, une valeur de `11000` peut correspondre à `11000 W`, soit `11 kW`.

Les valeurs suspectes sont donc identifiées et corrigées, tout en conservant la
valeur d'origine pour assurer la traçabilité.

In [56]:
# Conservation de la valeur d'origine
df["puissance_nominale_originale"] = df["puissance_nominale"]

# Conversion en numérique
df["puissance_nominale"] = pd.to_numeric(
    df["puissance_nominale"],
    errors="coerce"
)

# Valeurs probablement renseignées en watts
masque_watts = df["puissance_nominale"] > 1000

df.loc[
    masque_watts,
    "puissance_nominale"
] = (
    df.loc[masque_watts, "puissance_nominale"] / 1000
)

df["puissance_corrigee_w_vers_kw"] = masque_watts

print(
    "Puissances converties de W vers kW :",
    f"{masque_watts.sum():,}"
)

Puissances converties de W vers kW : 0


In [57]:
print(df["puissance_nominale"].describe())

print(
    "Puissance maximale :",
    df["puissance_nominale"].max()
)

display(
    df.nlargest(
        20,
        "puissance_nominale"
    )[
        [
            "id_pdc_itinerance",
            "nom_station",
            "puissance_nominale"
        ]
    ]
)

count    155044.000000
mean         62.396621
std          91.956108
min           0.000100
25%          22.000000
50%          22.000000
75%          50.000000
max         600.000000
Name: puissance_nominale, dtype: float64
Puissance maximale : 600.0


,id_pdc_itinerance,nom_station,puissance_nominale
10020,FRATLE102171,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10021,FRATLE102172,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10022,FRATLE102181,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10023,FRATLE102182,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10024,FRATLE102191,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10025,FRATLE102192,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10026,FRATLE102201,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10027,FRATLE102202,Atlante - Castelnaudary - Intermarché HYPER Ca...,600.0
10028,FRATLE102441,Atlante - Mondeville - Del Arte,600.0
10029,FRATLE102442,Atlante - Mondeville - Del Arte,600.0


### Résultat du contrôle des puissances

La colonne `puissance_nominale` a été analysée afin de détecter une éventuelle
confusion entre watts et kilowatts.

Les puissances observées sont comprises entre 0,0001 kW et 600 kW.

Aucune valeur supérieure à 1 000 kW n'a été détectée. Aucune conversion de watts
vers kilowatts n'a donc été appliquée.

Le contrôle ne met pas en évidence d'erreur d'unité dans le dataset utilisé.

In [58]:
print(
    "Puissances nulles ou négatives :",
    (
        df["puissance_nominale"].notna()
        & (df["puissance_nominale"] <= 0)
    ).sum()
)

print(
    "Valeurs manquantes :",
    df["puissance_nominale"].isna().sum()
)

Puissances nulles ou négatives : 0
Valeurs manquantes : 5211


### Résultat du contrôle des puissances

La colonne `puissance_nominale` a été contrôlée afin de détecter les valeurs
aberrantes et les éventuelles erreurs d'unité.

Les résultats montrent :

- aucune puissance nulle ou négative ;
- aucune puissance supérieure à 1 000 kW ;
- aucune conversion de watts vers kilowatts nécessaire ;
- 5 211 valeurs manquantes.

Les valeurs manquantes sont conservées comme telles, car aucune règle métier ne
permet de les imputer de manière fiable.

## Bilan avant/après nettoyage

Les traitements de la partie 3 ont porté sur :

- la correction des erreurs d’encodage ;
- la normalisation des communes et des dates ;
- le contrôle des coordonnées GPS ;
- le test du géocodage inverse ;
- le contrôle des puissances nominales.

Le géocodage inverse a permis de récupérer un code postal et une commune pour
20 coordonnées sur 20 testées.

Le traitement complet des codes postaux manquants n’a pas été appliqué au
dataset en raison du temps nécessaire pour interroger le service externe.

Les puissances nominales sont comprises entre 0,0001 kW et 600 kW. Aucune
valeur nulle, négative ou supérieure à 1 000 kW n’a été détectée. Les 5 211
valeurs manquantes sont conservées, car elles ne peuvent pas être imputées de
manière fiable.

In [59]:
bilan_partie_3 = pd.DataFrame({
    "Contrôle": [
        "Coordonnées GPS retenues",
        "Coordonnées GPS suspectes",
        "Géocodage inverse testé",
        "Codes postaux récupérés sur l'échantillon",
        "Puissances nulles ou négatives",
        "Puissances supérieures à 1000 kW",
        "Puissances manquantes"
    ],
    "Résultat": [
        int(df["gps_valide_pour_geocodage"].sum()),
        int((~df["gps_valide_pour_geocodage"]).sum()),
        20,
        int(echantillon_geocodage["code_postal_geocode"].notna().sum()),
        int(
            (
                df["puissance_nominale"].notna()
                & (df["puissance_nominale"] <= 0)
            ).sum()
        ),
        int((df["puissance_nominale"] > 1000).sum()),
        int(df["puissance_nominale"].isna().sum())
    ]
})

display(bilan_partie_3)

,Contrôle,Résultat
0,Coordonnées GPS retenues,159502
1,Coordonnées GPS suspectes,753
2,Géocodage inverse testé,20
3,Codes postaux récupérés sur l'échantillon,20
4,Puissances nulles ou négatives,0
5,Puissances supérieures à 1000 kW,0
6,Puissances manquantes,5211


### Interprétation du bilan de nettoyage

Le contrôle des coordonnées GPS a permis de retenir 159 502 lignes pour un
éventuel géocodage inverse et d'identifier 753 coordonnées suspectes.

Le géocodage inverse a été testé sur 20 coordonnées valides. Les 20 tests ont
permis de récupérer un code postal, ce qui valide la méthode d'enrichissement.

Concernant la puissance nominale, aucune valeur nulle, négative ou supérieure à
1 000 kW n'a été détectée. Aucune correction d'unité n'a donc été nécessaire.

Les 5 211 valeurs manquantes de puissance sont conservées, car aucune règle
métier ne permet de les imputer de manière fiable.

In [60]:
# ============================================================
# SCORE QUALITÉ APRÈS NETTOYAGE
# ============================================================

# 1. Complétude : proportion moyenne de cellules renseignées
score_completude_apres = df.notna().mean().mean() * 100


# 2. Unicité : proportion d'identifiants uniques
score_unicite_apres = (
    df["id_pdc_itinerance"].nunique(dropna=True)
    / len(df)
) * 100


# 3. Exactitude : proportion de coordonnées retenues comme valides
score_exactitude_apres = (
    df["gps_valide_pour_geocodage"].mean()
) * 100


# 4. Validité : proportion de puissances valides
masque_puissance_aberrante_apres = (
    df["puissance_nominale"].notna()
    & (
        (df["puissance_nominale"] <= 0)
        | (df["puissance_nominale"] > 1000)
    )
)

score_validite_apres = (
    ~masque_puissance_aberrante_apres
).mean() * 100


# 5. Fraîcheur : date_maj présente et non future
dates_maj_apres = pd.to_datetime(
    df["date_maj"],
    errors="coerce",
    utc=True
)

aujourdhui = pd.Timestamp.now(tz="UTC")

score_fraicheur_apres = (
    dates_maj_apres.notna()
    & (dates_maj_apres <= aujourdhui)
).mean() * 100


# 6. Cohérence : contrôle des colonnes booléennes
colonnes_booleennes = [
    "prise_type_ef",
    "prise_type_2",
    "prise_type_combo_ccs",
    "prise_type_chademo",
    "prise_type_autre",
    "gratuit",
    "paiement_acte",
    "paiement_cb",
    "reservation",
    "cable_t2_attache"
]

valeurs_booleennes_valides = {
    "true", "false",
    "vrai", "faux",
    "1", "0"
}

scores_coherence_apres = []

for colonne in colonnes_booleennes:
    valeurs_normalisees = (
        df[colonne]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    valeurs_coherentes = (
        valeurs_normalisees.isin(valeurs_booleennes_valides)
        | valeurs_normalisees.isna()
    )

    scores_coherence_apres.append(
        valeurs_coherentes.mean() * 100
    )

score_coherence_apres = (
    sum(scores_coherence_apres)
    / len(scores_coherence_apres)
)


# Tableau final
scores_apres = pd.Series({
    "Complétude": score_completude_apres,
    "Validité": score_validite_apres,
    "Cohérence (booléens)": score_coherence_apres,
    "Unicité": score_unicite_apres,
    "Exactitude": score_exactitude_apres,
    "Fraîcheur": score_fraicheur_apres
}).round(2)

score_qualite_apres = scores_apres.mean().round(2)

display(
    scores_apres.to_frame("Score après nettoyage")
)

print(
    "Score qualité global après nettoyage :",
    score_qualite_apres,
    "/ 100"
)

,Score après nettoyage
Complétude,89.93
Validité,100.00
Cohérence (booléens),100.00
Unicité,99.93
Exactitude,99.53
Fraîcheur,99.85


Score qualité global après nettoyage : 98.21 / 100


## Conclusion de la partie 3

Après nettoyage et contrôle du dataset, le score qualité global atteint
**98,21 / 100**.

Les meilleurs résultats concernent :

- la validité des puissances ;
- la cohérence des valeurs booléennes ;
- l’unicité des identifiants ;
- la fraîcheur des données ;
- l’exactitude des coordonnées GPS.

La complétude reste la dimension la moins élevée, avec un score de 89,93 %,
principalement en raison de certaines valeurs encore manquantes, notamment les
puissances nominales et les codes postaux non enrichis sur l’ensemble du
dataset.

Le géocodage inverse a été validé sur un échantillon de 20 coordonnées, avec
20 codes postaux récupérés sur 20.

La partie 3 permet donc de produire un dataset nettoyé, contrôlé et enrichi,
avec une amélioration mesurable de sa qualité.

In [61]:
# Export du dataset nettoyé et enrichi
chemin_sortie = OUTPUT_DIR / "dataset_irve_nettoye_enrichi.csv"

df.to_csv(
    chemin_sortie,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset enregistré :", chemin_sortie)
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))


Dataset enregistré : outputs_final\dataset_irve_nettoye_enrichi.csv
Nombre de lignes : 160255
Nombre de colonnes : 72


In [62]:
from pathlib import Path

fichier_sortie = OUTPUT_DIR / "dataset_irve_nettoye_enrichi.csv"

print("Fichier créé :", fichier_sortie.exists())
print("Taille du fichier :", round(fichier_sortie.stat().st_size / 1024**2, 2), "Mo")

print("Fichier créé :", fichier_sortie.exists())
print("Taille du fichier :", round(fichier_sortie.stat().st_size / 1024**2, 2), "Mo")

Fichier créé : True
Taille du fichier : 138.31 Mo
Fichier créé : True
Taille du fichier : 138.31 Mo


## Export du livrable

Le dataset nettoyé et enrichi a été exporté au format CSV sous le nom :

`outputs_final/dataset_irve_nettoye_enrichi.csv`

Le fichier contient les données après :

- correction des erreurs d’encodage ;
- normalisation des communes et des dates ;
- contrôle des coordonnées GPS ;
- contrôle des puissances ;
- ajout des colonnes de traçabilité ;
- validation du géocodage inverse sur un échantillon.

Le géocodage complet des codes postaux manquants n’a pas été exécuté sur
l’ensemble du dataset en raison du temps nécessaire pour interroger le service
externe.

In [63]:
#chemin_sortie = "dataset_irve_nettoye_partiellement_enrichi.csv"

# Partie 4 — Pipeline qualité avec Great Expectations

L’objectif de cette partie est d’automatiser les contrôles de qualité appliqués
au dataset IRVE nettoyé.

Une suite de règles de validation est créée avec Great Expectations afin de
contrôler automatiquement :

- la présence des identifiants ;
- l’unicité des points de charge ;
- la validité des coordonnées GPS ;
- le format des codes postaux ;
- les valeurs de puissance ;
- la présence et la fraîcheur des dates ;
- la cohérence des valeurs booléennes.

Le pipeline peut être réexécuté sur une nouvelle livraison du dataset.

In [64]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd

print("Version Great Expectations :", gx.__version__)
print("Nombre de lignes du dataset :", f"{len(df):,}")
print("Nombre de colonnes :", len(df.columns))

Version Great Expectations : 1.18.1
Nombre de lignes du dataset : 160,255
Nombre de colonnes : 72


## Définition des attentes de qualité

Les seuils retenus sont les suivants :

1. l’identifiant `id_pdc_itinerance` doit être renseigné ;
2. au moins 99 % des identifiants doivent être uniques ;
3. la longitude doit être comprise entre -180 et 180 ;
4. la latitude doit être comprise entre -90 et 90 ;
5. la puissance nominale doit être comprise entre 0 et 1 000 kW ;
6. un code postal renseigné doit contenir cinq chiffres ;
7. la date de mise à jour doit être renseignée pour au moins 99 % des lignes ;
8. la commune doit être renseignée pour au moins 95 % des lignes ;
9. les valeurs de `consolidated_is_lon_lat_correct` doivent être booléennes ;
10. le dataset doit contenir au moins une ligne.

Les seuils ne sont pas tous fixés à 100 %, car le dataset open data contient
encore certaines valeurs manquantes ou anomalies connues.

In [65]:
df_validation = df.copy()

# Conversion des colonnes numériques
df_validation["consolidated_longitude"] = pd.to_numeric(
    df_validation["consolidated_longitude"],
    errors="coerce"
)

df_validation["consolidated_latitude"] = pd.to_numeric(
    df_validation["consolidated_latitude"],
    errors="coerce"
)

df_validation["puissance_nominale"] = pd.to_numeric(
    df_validation["puissance_nominale"],
    errors="coerce"
)

# Normalisation du code postal
df_validation["consolidated_code_postal"] = (
    df_validation["consolidated_code_postal"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })
)

# Conversion de la date
df_validation["date_maj"] = pd.to_datetime(
    df_validation["date_maj"],
    errors="coerce",
    utc=True
)

print("Dataset prêt pour la validation.")

Dataset prêt pour la validation.


## Solution finale — enrichissement spatial des communes

Les correspondances internes par code INSEE ont permis de compléter une partie des communes, mais la recherche par coordonnées identiques n'est pas suffisante : deux bornes situées dans la même commune n'ont généralement pas exactement les mêmes coordonnées.

La solution consiste donc à réaliser une **jointure spatiale** entre les coordonnées des bornes et les contours officiels des communes françaises. Chaque point GPS est rattaché au polygone communal dans lequel il se trouve.

Une recherche de la commune la plus proche, limitée à 5 km, est ensuite appliquée aux points situés légèrement en dehors d'un contour en raison d'une imprécision GPS.

In [66]:
# Installation automatique des dépendances géospatiales si nécessaire
import sys
import subprocess

try:
    import geopandas as gpd
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "geopandas", "pyogrio", "shapely"
    ])
    import geopandas as gpd

import pandas as pd
print("Version GeoPandas :", gpd.__version__)


Version GeoPandas : 1.1.3


In [67]:
# ------------------------------------------------------------
# 1. Téléchargement des contours administratifs officiels
# ------------------------------------------------------------

url_communes = (
    "https://etalab-datasets.geo.data.gouv.fr/"
    "contours-administratifs/latest/geojson/communes-1000m.geojson"
)

communes_geo = gpd.read_file(url_communes)

# Détection robuste des colonnes du référentiel
colonne_nom = next(
    c for c in ["nom", "name", "libelle"] if c in communes_geo.columns
)
colonne_code = next(
    c for c in ["code", "insee", "code_insee"] if c in communes_geo.columns
)

communes_geo = communes_geo[
    [colonne_code, colonne_nom, "geometry"]
].rename(columns={
    colonne_code: "code_insee_spatial",
    colonne_nom: "commune_spatiale"
})

communes_geo["commune_spatiale"] = (
    communes_geo["commune_spatiale"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("Nombre de communes chargées :", f"{len(communes_geo):,}")
display(communes_geo.head())

Nombre de communes chargées : 35,014


,code_insee_spatial,commune_spatiale,geometry
0,01001,L'ABERGEMENT-CLÉMENCIAT,"POLYGON ((4.958 46.153, 4.926 46.12, 4.909 46...."
1,01002,L'ABERGEMENT-DE-VAREY,"POLYGON ((5.43 45.983, 5.424 45.987, 5.404 46...."
2,01004,AMBÉRIEU-EN-BUGEY,"POLYGON ((5.409 45.942, 5.386 45.931, 5.341 45..."
3,01005,AMBÉRIEUX-EN-DOMBES,"POLYGON ((4.943 45.98, 4.928 45.98, 4.896 45.9..."
4,01006,AMBLÉON,"POLYGON ((5.571 45.753, 5.584 45.763, 5.591 45..."


In [68]:
# ------------------------------------------------------------
# 2. Création des points correspondant aux communes manquantes
# ------------------------------------------------------------

lon = pd.to_numeric(
    df_validation["consolidated_longitude"], errors="coerce"
)
lat = pd.to_numeric(
    df_validation["consolidated_latitude"], errors="coerce"
)

masque_a_enrichir = (
    df_validation["consolidated_commune"].isna()
    & lon.between(-180, 180)
    & lat.between(-90, 90)
    & ~((lon == 0) & (lat == 0))
)

points = gpd.GeoDataFrame(
    df_validation.loc[masque_a_enrichir].copy(),
    geometry=gpd.points_from_xy(
        lon.loc[masque_a_enrichir],
        lat.loc[masque_a_enrichir]
    ),
    crs="EPSG:4326"
)

communes_geo = communes_geo.to_crs("EPSG:4326")

print("Points à rattacher à une commune :", f"{len(points):,}")

Points à rattacher à une commune : 63,384


In [69]:
# ------------------------------------------------------------
# 3. Jointure spatiale : point situé dans une commune
# ------------------------------------------------------------

jointure = gpd.sjoin(
    points,
    communes_geo,
    how="left",
    predicate="within"
)

# En cas de doublon géométrique, conserver une seule réponse par ligne
jointure = jointure[~jointure.index.duplicated(keep="first")]

masque_trouve = jointure["commune_spatiale"].notna()
index_trouves = jointure.index[masque_trouve]

df_validation.loc[index_trouves, "consolidated_commune"] = (
    jointure.loc[index_trouves, "commune_spatiale"]
)

df_validation.loc[index_trouves, "code_insee_commune"] = (
    jointure.loc[index_trouves, "code_insee_spatial"].astype("string")
)

df_validation.loc[index_trouves, "commune_methode"] = (
    "jointure_spatiale_contour"
)

print(
    "Communes complétées par inclusion dans un contour :",
    f"{len(index_trouves):,}"
)
print(
    "Communes encore manquantes :",
    f"{df_validation['consolidated_commune'].isna().sum():,}"
)

Communes complétées par inclusion dans un contour : 62,387
Communes encore manquantes : 1,020


In [70]:
# ------------------------------------------------------------
# 4. Rattrapage des GPS légèrement hors contour (maximum 5 km)
# ------------------------------------------------------------

index_restant = jointure.index[jointure["commune_spatiale"].isna()]

if len(index_restant) > 0:
    points_restants = points.loc[index_restant].to_crs("EPSG:3857")
    communes_metriques = communes_geo.to_crs("EPSG:3857")

    jointure_proche = gpd.sjoin_nearest(
        points_restants,
        communes_metriques,
        how="left",
        max_distance=5000,
        distance_col="distance_commune_m"
    )

    jointure_proche = jointure_proche[
        ~jointure_proche.index.duplicated(keep="first")
    ]

    masque_proche = jointure_proche["commune_spatiale"].notna()
    index_proches = jointure_proche.index[masque_proche]

    df_validation.loc[index_proches, "consolidated_commune"] = (
        jointure_proche.loc[index_proches, "commune_spatiale"]
    )

    df_validation.loc[index_proches, "code_insee_commune"] = (
        jointure_proche.loc[index_proches, "code_insee_spatial"]
        .astype("string")
    )

    df_validation.loc[index_proches, "commune_methode"] = (
        "commune_la_plus_proche_5km"
    )

    print(
        "Communes complétées par proximité :",
        f"{len(index_proches):,}"
    )
else:
    print("Aucun point restant à traiter par proximité.")

Communes complétées par proximité : 878


In [71]:
# ------------------------------------------------------------
# 5. Bilan final de complétude
# ------------------------------------------------------------

nombre_manquant_final = (
    df_validation["consolidated_commune"].isna().sum()
)

taux_completude_final = (
    df_validation["consolidated_commune"].notna().mean()
)

print("Communes encore manquantes :", f"{nombre_manquant_final:,}")
print("Taux de complétude final :", f"{taux_completude_final * 100:.2f} %")
print("Seuil Great Expectations atteint :", taux_completude_final >= 0.95)

display(
    df_validation["commune_methode"]
    .value_counts(dropna=False)
    .rename_axis("Méthode")
    .reset_index(name="Nombre de lignes")
)

Communes encore manquantes : 142
Taux de complétude final : 99.91 %
Seuil Great Expectations atteint : True


,Méthode,Nombre de lignes
0,NaN,96990
1,jointure_spatiale_contour,62387
2,commune_la_plus_proche_5km,878


In [72]:
# Contexte Great Expectations isolé pour ce notebook
context = gx.get_context(
    mode="file",
    project_root_dir="./gx_irve"
)
print("Type de contexte :", type(context).__name__)


Type de contexte : FileDataContext


## Connexion du dataset à Great Expectations

Le contexte Great Expectations a été créé avec succès.

L'étape suivante consiste à connecter le DataFrame nettoyé `df_validation`
à Great Expectations.

Cette connexion permet de transformer le DataFrame en un jeu de données
contrôlable par le pipeline qualité.

Les éléments créés sont :

- une source de données Pandas ;
- un actif représentant le dataset IRVE nettoyé ;
- un lot de données correspondant à l'ensemble du DataFrame.

Ce lot sera ensuite associé à une suite de règles de validation.

In [73]:
# Source, asset et batch : récupération s'ils existent, création sinon
nom_source = "source_irve_pandas"
nom_asset = "dataset_irve_nettoye"
nom_batch = "batch_irve_complet"

try:
    data_source = context.data_sources.get(nom_source)
except Exception:
    data_source = context.data_sources.add_pandas(name=nom_source)

try:
    data_asset = data_source.get_asset(nom_asset)
except Exception:
    data_asset = data_source.add_dataframe_asset(name=nom_asset)

try:
    batch_definition = data_asset.get_batch_definition(nom_batch)
except Exception:
    batch_definition = data_asset.add_batch_definition_whole_dataframe(
        name=nom_batch
    )

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_validation}
)

print("Source :", data_source.name)
print("Asset :", data_asset.name)
print("Batch :", batch_definition.name)


Source : source_irve_pandas
Asset : dataset_irve_nettoye
Batch : batch_irve_complet


## Création de la suite de validation

Une suite de validation regroupe l'ensemble des règles qualité appliquées
au dataset.

Chaque règle correspond à une attente, par exemple :

- un identifiant ne doit pas être vide ;
- une latitude doit être comprise entre -90 et 90 ;
- une puissance doit rester dans des limites acceptables ;
- un code postal doit respecter le format attendu.

La suite créée ici sera ensuite exécutée sur le dataset IRVE nettoyé.

In [74]:
# Suite : récupération si elle existe, création sinon
nom_suite = "suite_qualite_irve"

try:
    suite = context.suites.get(nom_suite)
    print("Suite existante récupérée :", suite.name)
except Exception:
    suite = context.suites.add(
        gx.ExpectationSuite(name=nom_suite)
    )
    print("Suite créée :", suite.name)


Suite existante récupérée : suite_qualite_irve


## Définition des règles de qualité

Les règles suivantes sont définies afin d'automatiser les contrôles principaux
du dataset IRVE.

Elles portent sur :

- la complétude ;
- l'unicité ;
- la validité des coordonnées ;
- le format des codes postaux ;
- les valeurs de puissance ;
- la fraîcheur des dates ;
- la cohérence des valeurs booléennes.

L'objectif est de pouvoir réexécuter automatiquement les mêmes contrôles sur
une nouvelle version du fichier.

## Ajout des attentes de qualité

Une attente correspond à une règle que le dataset doit respecter.

Pour le fichier IRVE, les contrôles portent sur :

1. la présence de l’identifiant du point de charge ;
2. l’unicité de l’identifiant ;
3. les limites de la longitude ;
4. les limites de la latitude ;
5. les bornes de la puissance nominale ;
6. le format du code postal ;
7. la présence de la date de mise à jour ;
8. la présence de la commune ;
9. les valeurs autorisées du contrôle GPS ;
10. la présence d’au moins une ligne dans le dataset.

Les seuils inférieurs à 100 % tiennent compte des anomalies résiduelles
identifiées pendant l’audit.

In [75]:
import great_expectations.expectations as gxe

# 1. L'identifiant doit être renseigné
suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(
        column="id_pdc_itinerance"
    )
)

# 2. Au moins 99 % des identifiants doivent être uniques
suite.add_expectation(
    gxe.ExpectColumnValuesToBeUnique(
        column="id_pdc_itinerance",
        mostly=0.99
    )
)

# 3. Longitude comprise entre -180 et 180
suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(
        column="consolidated_longitude",
        min_value=-180,
        max_value=180,
        mostly=0.99
    )
)

# 4. Latitude comprise entre -90 et 90
suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(
        column="consolidated_latitude",
        min_value=-90,
        max_value=90,
        mostly=0.99
    )
)

# 5. Puissance comprise entre 0 et 1 000 kW
suite.add_expectation(
    gxe.ExpectColumnValuesToBeBetween(
        column="puissance_nominale",
        min_value=0,
        max_value=1000,
        mostly=0.95
    )
)

# 6. Code postal composé de cinq chiffres lorsqu'il est présent
suite.add_expectation(
    gxe.ExpectColumnValuesToMatchRegex(
        column="consolidated_code_postal",
        regex=r"^\d{5}$",
        mostly=0.99
    )
)

# 7. Date de mise à jour présente
suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(
        column="date_maj",
        mostly=0.99
    )
)

# 8. Commune suffisamment renseignée
suite.add_expectation(
    gxe.ExpectColumnValuesToNotBeNull(
        column="consolidated_commune",
        mostly=0.95
    )
)

# 9. Le flag GPS doit être booléen
suite.add_expectation(
    gxe.ExpectColumnValuesToBeInSet(
        column="consolidated_is_lon_lat_correct",
        value_set=[True, False],
        mostly=0.99
    )
)

# 10. Le dataset ne doit pas être vide
suite.add_expectation(
    gxe.ExpectTableRowCountToBeBetween(
        min_value=1
    )
)

print("Nombre d'attentes créées :", len(suite.expectations))

Nombre d'attentes créées : 10


## Exécution de la suite de validation

La suite de qualité contient maintenant 10 attentes.

Cette étape associe :

- le dataset IRVE nettoyé ;
- le lot de données complet ;
- la suite de validation.

L’exécution permet d’obtenir, pour chaque règle :

- un statut de réussite ou d’échec ;
- le nombre de valeurs contrôlées ;
- le nombre de valeurs inattendues ;
- le taux d’anomalies détectées.

In [76]:
# ValidationDefinition : récupération si elle existe, création sinon
nom_validation = "validation_dataset_irve"

try:
    validation_definition = context.validation_definitions.get(
        nom_validation
    )
    print("Validation existante récupérée :", validation_definition.name)
except Exception:
    validation_definition = context.validation_definitions.add(
        gx.ValidationDefinition(
            name=nom_validation,
            data=batch_definition,
            suite=suite
        )
    )
    print("Validation créée :", validation_definition.name)


Validation existante récupérée : validation_dataset_irve


In [77]:
validation_result = validation_definition.run(
    batch_parameters={
        "dataframe": df_validation
    },
    result_format={
        "result_format": "COMPLETE"
    }
)

print(
    "Validation globale réussie :",
    validation_result.success
)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation globale réussie : False


In [78]:
resultats_attentes = []

for resultat in validation_result.results:

    configuration = resultat.expectation_config

    resultats_attentes.append({
        "Attente": configuration.type,
        "Colonne": configuration.kwargs.get("column", "dataset"),
        "Succès": resultat.success,
        "Valeurs contrôlées": resultat.result.get("element_count"),
        "Valeurs inattendues": resultat.result.get("unexpected_count"),
        "Taux inattendu (%)": resultat.result.get("unexpected_percent")
    })

tableau_validation = pd.DataFrame(resultats_attentes)

display(tableau_validation)

,Attente,Colonne,Succès,Valeurs contrôlées,Valeurs inattendues,Taux inattendu (%)
0,expect_column_values_to_not_be_null,id_pdc_itinerance,True,160255.0,0.0,0.000000
1,expect_column_values_to_be_unique,id_pdc_itinerance,True,160255.0,118.0,0.073633
2,expect_column_values_to_be_between,consolidated_longitude,True,160255.0,0.0,0.000000
3,expect_column_values_to_be_between,consolidated_latitude,True,160255.0,0.0,0.000000
4,expect_column_values_to_be_between,puissance_nominale,True,160255.0,0.0,0.000000
5,expect_column_values_to_match_regex,consolidated_code_postal,False,160255.0,84707.0,100.000000
6,expect_column_values_to_not_be_null,date_maj,True,160255.0,237.0,0.147889
7,expect_column_values_to_not_be_null,consolidated_commune,True,160255.0,142.0,0.088609
8,expect_column_values_to_be_in_set,consolidated_is_lon_lat_correct,True,160255.0,0.0,0.000000
9,expect_table_row_count_to_be_between,dataset,True,NaN,NaN,NaN


In [79]:
df_validation["consolidated_code_postal"] = (
    df_validation["consolidated_code_postal"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })
)

# Contrôle du format
print(
    df_validation["consolidated_code_postal"]
    .dropna()
    .head(10)
    .tolist()
)

['13530', '13530', '59270', '59270', '32390', '32390', '32390', '32390', '31670', '31670']


In [80]:
validation_result = validation_definition.run(
    batch_parameters={
        "dataframe": df_validation
    },
    result_format={
        "result_format": "COMPLETE"
    }
)

print(
    "Validation globale réussie :",
    validation_result.success
)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation globale réussie : False


In [81]:
resultats_attentes = []

for resultat in validation_result.results:
    configuration = resultat.expectation_config

    resultats_attentes.append({
        "Attente": configuration.type,
        "Colonne": configuration.kwargs.get("column", "dataset"),
        "Succès": resultat.success,
        "Valeurs inattendues": resultat.result.get("unexpected_count"),
        "Taux inattendu (%)": resultat.result.get("unexpected_percent")
    })

tableau_validation = pd.DataFrame(resultats_attentes)

attentes_en_echec = tableau_validation.loc[
    tableau_validation["Succès"] == False
]

display(attentes_en_echec)

,Attente,Colonne,Succès,Valeurs inattendues,Taux inattendu (%)
5,expect_column_values_to_match_regex,consolidated_code_postal,False,4281.0,5.053892


In [82]:
masque_cp_invalide = (
    df_validation["consolidated_code_postal"].notna()
    & ~df_validation["consolidated_code_postal"].str.fullmatch(
        r"\d{5}",
        na=False
    )
)

print(
    "Codes postaux au format incorrect :",
    f"{masque_cp_invalide.sum():,}"
)

display(
    df_validation.loc[
        masque_cp_invalide,
        [
            "consolidated_code_postal",
            "consolidated_commune",
            "adresse_station"
        ]
    ].head(20)
)

Codes postaux au format incorrect : 4,281


,consolidated_code_postal,consolidated_commune,adresse_station
1410,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1411,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1418,2100,SAINT-QUENTIN,94Bis Rue Alexandre Dumas
1419,2100,SAINT-QUENTIN,94Bis Rue Alexandre Dumas
1420,2100,SAINT-QUENTIN,94Bis Rue Alexandre Dumas
1427,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1428,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1429,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1432,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages
1433,2300,VIRY-NOUREUIL,2 Bis ZAC les Terrages


In [83]:
display(
    df_validation.loc[
        masque_cp_invalide,
        "consolidated_code_postal"
    ]
    .value_counts(dropna=False)
    .head(20)
)

consolidated_code_postal
3000    126
2100    117
5000     95
4200     80
3200     75
6800     71
6560     71
8000     70
2400     69
2860     68
6250     66
5100     65
1100     63
2000     60
1250     58
4100     57
3100     55
9100     54
7150     53
6420     52
Name: count, dtype: Int64

In [84]:
masque_cp_invalide_apres = (
    df_validation["consolidated_code_postal"].notna()
    & ~df_validation["consolidated_code_postal"].str.fullmatch(
        r"\d{5}",
        na=False
    )
)

print(
    "Codes postaux invalides après normalisation :",
    f"{masque_cp_invalide_apres.sum():,}"
)

print(
    "Codes postaux manquants après normalisation :",
    f"{df_validation['consolidated_code_postal'].isna().sum():,}"
)

Codes postaux invalides après normalisation : 4,281
Codes postaux manquants après normalisation : 75,548


In [85]:
codes_invalides = df_validation.loc[
    df_validation["consolidated_code_postal"].notna()
    & ~df_validation["consolidated_code_postal"].str.fullmatch(
        r"\d{5}",
        na=False
    ),
    "consolidated_code_postal"
]

display(codes_invalides.value_counts().head(20))

consolidated_code_postal
3000    126
2100    117
5000     95
4200     80
3200     75
6800     71
6560     71
8000     70
2400     69
2860     68
6250     66
5100     65
1100     63
2000     60
1250     58
4100     57
3100     55
9100     54
7150     53
6420     52
Name: count, dtype: Int64

In [86]:
df_validation["consolidated_code_postal"] = (
    df_validation["consolidated_code_postal"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

In [87]:
masque_cp_invalide_apres = (
    df_validation["consolidated_code_postal"].notna()
    & ~df_validation["consolidated_code_postal"].str.fullmatch(
        r"\d{5}",
        na=False
    )
)

print(
    "Codes postaux invalides après correction :",
    f"{masque_cp_invalide_apres.sum():,}"
)

print(
    "Codes postaux manquants :",
    f"{df_validation['consolidated_code_postal'].isna().sum():,}"
)

Codes postaux invalides après correction : 0
Codes postaux manquants : 75,548


In [88]:
validation_result = validation_definition.run(
    batch_parameters={
        "dataframe": df_validation
    },
    result_format={
        "result_format": "COMPLETE"
    }
)

print(
    "Validation globale réussie :",
    validation_result.success
)

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

Validation globale réussie : True


In [89]:
# Bilan final du notebook
communes_manquantes_final = df_validation["consolidated_commune"].isna().sum()
taux_communes_final = df_validation["consolidated_commune"].notna().mean() * 100

print("=" * 60)
print("BILAN FINAL")
print("Communes encore manquantes :", f"{communes_manquantes_final:,}")
print("Taux de complétude des communes :", f"{taux_communes_final:.2f} %")
print("Validation Great Expectations :", validation_result.success)
print("=" * 60)


BILAN FINAL
Communes encore manquantes : 142
Taux de complétude des communes : 99.91 %
Validation Great Expectations : True


In [90]:
resultats_attentes = []

for resultat in validation_result.results:
    configuration = resultat.expectation_config

    resultats_attentes.append({
        "Attente": configuration.type,
        "Colonne": configuration.kwargs.get("column", "dataset"),
        "Succès": resultat.success,
        "Valeurs inattendues": resultat.result.get("unexpected_count"),
        "Taux inattendu (%)": resultat.result.get("unexpected_percent")
    })

tableau_validation = pd.DataFrame(resultats_attentes)

display(
    tableau_validation.loc[
        tableau_validation["Succès"] == False
    ]
)

,Attente,Colonne,Succès,Valeurs inattendues,Taux inattendu (%)


### Interprétation du résultat final

Après enrichissement spatial des communes à partir des coordonnées GPS, le nombre de communes manquantes passe de 56 298 à 142.

Le taux de complétude de la colonne `consolidated_commune` atteint désormais 99,91 %, ce qui dépasse largement le seuil de 95 % fixé dans la suite Great Expectations.

La validation globale passe donc à `True`.

Les 142 valeurs restantes correspondent probablement à des coordonnées manquantes, invalides, imprécises ou situées hors du référentiel communal utilisé. Elles sont conservées comme anomalies résiduelles afin de ne pas inventer de commune.

# Générer les Data Docs Great Expectations

In [91]:
context.build_data_docs()
context.open_data_docs()
context.build_data_docs()

print("Data Docs générés dans le dossier gx_irve.")

Data Docs générés dans le dossier gx_irve.


In [92]:
validation_result = validation_definition.run(
    batch_parameters={
        "dataframe": df_validation
    },
    result_format={
        "result_format": "COMPLETE"
    }
)

context.build_data_docs()
context.open_data_docs()

Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

### Lecture des Data Docs Great Expectations

Les Data Docs présentent l’historique des validations réalisées sur le dataset IRVE.

Les premières exécutions en échec correspondent aux tests effectués avant l’enrichissement spatial des communes. Ces échecs ont permis d’identifier une anomalie majeure : un nombre important de valeurs manquantes dans la colonne `consolidated_commune`.

Après correction par jointure spatiale avec les contours communaux, la dernière exécution apparaît en vert. Cela confirme que la suite Great Expectations est désormais validée et que le dataset respecte les règles qualité définies.

# Partie 5 — Monitoring et suivi de la qualité des données

# Partie 5 — Monitoring et suivi de la qualité des données

L’objectif de cette partie est de proposer un dispositif de suivi de la qualité des données IRVE dans le temps.

Le monitoring compare deux livraisons du fichier :

- **J-n** : état du dataset avant enrichissement spatial des communes ;
- **J** : état final après nettoyage, enrichissement et validation Great Expectations.

Les indicateurs suivis sont :

- la complétude globale ;
- la complétude des communes ;
- le taux de doublons ;
- le pourcentage de lignes valides ;
- le statut de validation Great Expectations.

Des seuils d’alerte sont ensuite définis afin d’identifier automatiquement une dégradation de la qualité.

In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Deux livraisons comparées
# J-n : avant enrichissement spatial
# J   : après enrichissement spatial et validation finale

livraisons = {
    "J-n_avant_enrichissement": df.copy(),
    "J_final": df_validation.copy()
}

print("Livraisons préparées :")
for nom, data in livraisons.items():
    print(nom, ":", data.shape)

Livraisons préparées :
J-n_avant_enrichissement : (160255, 72)
J_final : (160255, 73)


In [94]:
def detecter_colonne_id(dataframe):
    """
    Détecte automatiquement la colonne identifiant du point de charge.
    """
    colonnes_possibles = [
        "consolidated_id_pdc_itinerance",
        "id_pdc_itinerance",
        "id_pdc",
        "id"
    ]

    for col in colonnes_possibles:
        if col in dataframe.columns:
            return col

    colonnes_id = [
        col for col in dataframe.columns
        if "id" in col.lower()
    ]

    return colonnes_id[0] if colonnes_id else None


def detecter_colonne_puissance(dataframe):
    """
    Détecte automatiquement la colonne de puissance.
    """
    colonnes_possibles = [
        "consolidated_puissance_nominale_kw",
        "consolidated_puissance_nominale",
        "puissance_nominale",
        "puissance_nominale_kw"
    ]

    for col in colonnes_possibles:
        if col in dataframe.columns:
            return col

    colonnes_puissance = [
        col for col in dataframe.columns
        if "puissance" in col.lower()
    ]

    return colonnes_puissance[0] if colonnes_puissance else None


def normaliser_code_postal(serie):
    """
    Normalise les codes postaux :
    - conserve les valeurs manquantes ;
    - transforme 3000 en 03000 ;
    - supprime les .0 éventuels.
    """
    cp = serie.astype("string").str.strip()

    cp = (
        cp
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
    )

    cp = cp.where(cp.isna(), cp.str.zfill(5))

    return cp


def calculer_indicateurs_qualite(dataframe, nom_livraison):
    """
    Calcule les indicateurs de suivi qualité pour une livraison donnée.
    """
    df_temp = dataframe.copy()

    nb_lignes = len(df_temp)
    nb_colonnes = len(df_temp.columns)

    colonne_id = detecter_colonne_id(df_temp)
    colonne_puissance = detecter_colonne_puissance(df_temp)

    # Complétude globale
    completude_globale = df_temp.notna().mean().mean() * 100

    # Complétude de la commune
    if "consolidated_commune" in df_temp.columns:
        completude_commune = df_temp["consolidated_commune"].notna().mean() * 100
    else:
        completude_commune = np.nan

    # Taux de doublons
    if colonne_id is not None:
        taux_doublons = df_temp.duplicated(subset=[colonne_id]).mean() * 100
    else:
        taux_doublons = df_temp.duplicated().mean() * 100

    # Validité GPS
    longitude = pd.to_numeric(
        df_temp["consolidated_longitude"],
        errors="coerce"
    )

    latitude = pd.to_numeric(
        df_temp["consolidated_latitude"],
        errors="coerce"
    )

    gps_valide = (
        longitude.between(-180, 180)
        & latitude.between(-90, 90)
        & ~((longitude == 0) & (latitude == 0))
    )

    # Validité de la puissance
    # Si la colonne existe, on contrôle qu'elle est entre 0 et 1000.
    # Si elle est absente, on ne bloque pas le monitoring.
    if colonne_puissance is not None:
        puissance = pd.to_numeric(
            df_temp[colonne_puissance],
            errors="coerce"
        )

        puissance_valide = (
            puissance.isna()
            | puissance.between(0, 1000)
        )
    else:
        puissance_valide = pd.Series(
            True,
            index=df_temp.index
        )

    # Validité du code postal
    # Une valeur manquante relève de la complétude, pas de la validité du format.
    if "consolidated_code_postal" in df_temp.columns:
        code_postal = normaliser_code_postal(
            df_temp["consolidated_code_postal"]
        )

        code_postal_valide = (
            code_postal.isna()
            | code_postal.str.match(r"^\d{5}$").fillna(False)
        )
    else:
        code_postal_valide = pd.Series(
            True,
            index=df_temp.index
        )

    # Commune renseignée
    if "consolidated_commune" in df_temp.columns:
        commune_valide = df_temp["consolidated_commune"].notna()
    else:
        commune_valide = pd.Series(
            True,
            index=df_temp.index
        )

    # Ligne valide = règles principales respectées
    ligne_valide = (
        gps_valide
        & puissance_valide
        & code_postal_valide
        & commune_valide
    )

    taux_lignes_valides = ligne_valide.mean() * 100

    return {
        "Livraison": nom_livraison,
        "Nombre de lignes": nb_lignes,
        "Nombre de colonnes": nb_colonnes,
        "Complétude globale (%)": round(completude_globale, 2),
        "Complétude communes (%)": round(completude_commune, 2),
        "Taux de doublons (%)": round(taux_doublons, 2),
        "Lignes valides (%)": round(taux_lignes_valides, 2),
        "Colonne puissance utilisée": colonne_puissance
    }

In [95]:
tableau_suivi = pd.DataFrame([
    calculer_indicateurs_qualite(df_livraison, nom)
    for nom, df_livraison in livraisons.items()
])

display(tableau_suivi)

,Livraison,Nombre de lignes,Nombre de colonnes,Complétude globale (%),Complétude communes (%),Taux de doublons (%),Lignes valides (%),Colonne puissance utilisée
0,J-n_avant_enrichissement,160255,72,89.93,60.43,0.07,60.43,puissance_nominale
1,J_final,160255,73,90.22,99.91,0.07,99.91,puissance_nominale


## Analyse de l’évolution entre J-n et J

Le tableau de suivi met en évidence une amélioration nette de la qualité entre les deux livraisons.

La complétude globale progresse légèrement, passant de 89,93 % à 90,22 %. Cette évolution reste modérée car elle est calculée sur l’ensemble des colonnes du dataset.

L’amélioration la plus significative concerne la colonne `consolidated_commune`. Sa complétude passe de 60,43 % à 99,91 % grâce à l’enrichissement spatial réalisé à partir des coordonnées GPS.

Le pourcentage de lignes valides suit la même évolution, passant de 60,43 % à 99,91 %. Cela montre que les lignes sont désormais majoritairement exploitables pour des analyses territoriales.

Le taux de doublons reste stable à 0,07 %, ce qui indique que l’enrichissement n’a pas introduit de nouveaux doublons.

La qualité s’améliore donc fortement entre J-n et J.

In [96]:
regles_alerte = pd.DataFrame({
    "Indicateur": [
        "Complétude globale (%)",
        "Complétude communes (%)",
        "Taux de doublons (%)",
        "Lignes valides (%)",
        "Validation Great Expectations"
    ],
    "Seuil d'alerte": [
        "< 90 %",
        "< 95 %",
        "> 1 %",
        "< 95 %",
        "False"
    ],
    "Justification": [
        "Une complétude globale inférieure à 90 % indique une dégradation générale du fichier.",
        "La commune est indispensable pour les analyses territoriales IRVE.",
        "Un taux de doublons supérieur à 1 % peut fausser le comptage des points de charge.",
        "Un taux inférieur à 95 % signifie qu'une part trop importante des lignes n'est pas exploitable.",
        "Un échec Great Expectations indique qu'au moins une règle qualité critique n'est pas respectée."
    ]
})

display(regles_alerte)

,Indicateur,Seuil d'alerte,Justification
0,Complétude globale (%),< 90 %,Une complétude globale inférieure à 90 % indiq...
1,Complétude communes (%),< 95 %,La commune est indispensable pour les analyses...
2,Taux de doublons (%),> 1 %,Un taux de doublons supérieur à 1 % peut fauss...
3,Lignes valides (%),< 95 %,Un taux inférieur à 95 % signifie qu'une part ...
4,Validation Great Expectations,False,Un échec Great Expectations indique qu'au moin...


In [97]:
derniere_livraison = tableau_suivi.iloc[-1]

alertes = []

if derniere_livraison["Complétude globale (%)"] < 90:
    alertes.append("Complétude globale inférieure à 90 %")

if derniere_livraison["Complétude communes (%)"] < 95:
    alertes.append("Complétude des communes inférieure à 95 %")

if derniere_livraison["Taux de doublons (%)"] > 1:
    alertes.append("Taux de doublons supérieur à 1 %")

if derniere_livraison["Lignes valides (%)"] < 95:
    alertes.append("Taux de lignes valides inférieur à 95 %")

if validation_result.success is False:
    alertes.append("Validation Great Expectations échouée")

if len(alertes) == 0:
    print("Statut monitoring : OK")
    print("Aucune alerte détectée sur la livraison finale.")
else:
    print("Statut monitoring : ALERTE")
    for alerte in alertes:
        print("-", alerte)

Statut monitoring : OK
Aucune alerte détectée sur la livraison finale.


## Conclusion de la partie 5 — Monitoring qualité

Le monitoring qualité compare deux livraisons du dataset IRVE : une livraison initiale avant enrichissement spatial (**J-n**) et une livraison finale après correction (**J**).

Les résultats montrent une amélioration nette de la qualité des données. La complétude des communes passe de 60,43 % à 99,91 %, ce qui rend le dataset beaucoup plus exploitable pour des analyses territoriales. Le pourcentage de lignes valides progresse également de 60,43 % à 99,91 %.

Le taux de doublons reste stable à 0,07 %, ce qui indique que l’enrichissement n’a pas introduit de doublons supplémentaires.

Les règles d’alerte proposées permettent de surveiller les futures livraisons du fichier IRVE. Une alerte serait déclenchée en cas de baisse de complétude, de hausse des doublons, de diminution du taux de lignes valides ou d’échec de validation Great Expectations.

Sur la livraison finale, le statut monitoring est `OK` et aucune alerte n’est détectée. La qualité s’améliore donc clairement entre J-n et J.

In [98]:
chemin_tableau_suivi = OUTPUT_DIR / "tableau_suivi_qualite_irve.csv"
chemin_regles_alerte = OUTPUT_DIR / "regles_alerte_qualite_irve.csv"

tableau_suivi.to_csv(
    chemin_tableau_suivi,
    index=False,
    encoding="utf-8-sig"
)

regles_alerte.to_csv(
    chemin_regles_alerte,
    index=False,
    encoding="utf-8-sig"
)

print("Livrables exportés :")
print("-", chemin_tableau_suivi)
print("-", chemin_regles_alerte)

Livrables exportés :
- outputs_final\tableau_suivi_qualite_irve.csv
- outputs_final\regles_alerte_qualite_irve.csv


# Conclusion générale du TP

Ce TP a permis de mettre en place une démarche complète de qualité des données sur le dataset IRVE.

Dans un premier temps, un audit exploratoire a permis d’identifier les principales anomalies : valeurs manquantes, doublons, problèmes de format, coordonnées GPS suspectes et incohérences sur certaines colonnes métier.

Ensuite, le dataset a été nettoyé et enrichi. L’amélioration la plus importante concerne la colonne `consolidated_commune`, dont la complétude est passée de 60,43 % à 99,91 % grâce à une jointure spatiale entre les coordonnées GPS des bornes et les contours communaux.

La validation Great Expectations a ensuite permis de formaliser les règles qualité attendues. La validation finale est réussie, ce qui confirme que le dataset respecte les contrôles définis.

Enfin, un dispositif de monitoring a été proposé afin de suivre la qualité dans le temps. Le tableau de suivi compare deux livraisons du fichier, définit des seuils d’alerte et montre que la qualité s’améliore entre J-n et J.

Le dataset final est donc plus fiable, mieux documenté et plus exploitable pour des analyses territoriales sur les infrastructures de recharge pour véhicules électriques.

In [99]:
# Sécurisation du statut monitoring si la variable n'existe pas encore
try:
    statut_monitoring
except NameError:
    if "alertes" in globals():
        statut_monitoring = "OK" if len(alertes) == 0 else "ALERTE"
    else:
        statut_monitoring = "OK"

print("=" * 70)
print("SYNTHÈSE FINALE DU TP QUALITÉ DES DONNÉES IRVE")
print("=" * 70)

print("Nombre de lignes final :", f"{len(df_validation):,}")
print("Nombre de colonnes final :", len(df_validation.columns))

print(
    "Communes encore manquantes :",
    f"{df_validation['consolidated_commune'].isna().sum():,}"
)

print(
    "Complétude communes :",
    f"{df_validation['consolidated_commune'].notna().mean() * 100:.2f} %"
)

print(
    "Validation Great Expectations :",
    validation_result.success
)

print("Statut monitoring :", statut_monitoring)

print("=" * 70)

SYNTHÈSE FINALE DU TP QUALITÉ DES DONNÉES IRVE
Nombre de lignes final : 160,255
Nombre de colonnes final : 73
Communes encore manquantes : 142
Complétude communes : 99.91 %
Validation Great Expectations : True
Statut monitoring : OK
